# Program Skripsi v4 — Analisis Sentimen Ulasan Wondr by BNI

**Optimasi *Hyperparameter* Model IndoBERT Menggunakan *Particle Swarm Optimization* Disertai Uji Signifikansi Statistik pada Analisis Sentimen Ulasan Aplikasi Wondr by BNI**

Notebook Google Colab (GPU **T4**) dengan folder kerja `MyDrive/skripsi_wondr_v4/`. Isinya hanya program inti. Sel pemeriksaan tambahan (audit) tetap tersimpan sebagai arsip di `skripsi_wondr_v4.ipynb`.

### Cara pakai
1. *Runtime → Change runtime type →* **T4 GPU**.
2. Di setiap sesi baru, jalankan **Tahap 0.1** (2 sel) lalu **Tahap 0.2** (2 sel).
3. Setelah itu, langsung jalankan tahap yang sedang dikerjakan. **Jangan memakai *Run all*.**
4. **Tahap 1–9 sudah final**, termasuk anotasi. Selnya disimpan sebagai catatan program dan **tidak dijalankan ulang** (`REVISI_12NOV.md`, aturan 5). Hasilnya sudah ada di `hasil/`, `anotasi/`, `log/`, dan `bab4/`.
5. Sel `%%writefile lib/...` hanya menulis berkas fungsi ke Drive. Berkasnya sudah ada, jadi tidak perlu dijalankan lagi kecuali isinya diubah.

### Daftar isi dan status
| Bagian | Tahap | GPU | Status |
|---|---|---|---|
| A. Persiapan | 0.1 Lingkungan · 0.2 Konfigurasi | – | jalankan setiap sesi |
| B. Data | 1.1 Muat data · 1.2 Audit kualitas data · 2 Seleksi atribut · 3 Pelabelan awal · 4 *Pre-processing* · 5 Deduplikasi · 6 Pembagian data · 7 Tokenisasi | – | ✅ final |
| C. Label | 8.1 Kode pelatihan · 8.2 *Confident learning* · 8.3 Kandidat label salah · 9.1–9.3 Anotasi manual · 9.4 Eksplorasi data | 8.1, 8.2 | ✅ final |
| D. Pelatihan | 10 *Fine-tuning* skema *default* | ✔ | ✅ FP32 (*fitness* 0,9276) |
| E. Menyusul | 10.0 Uji AMP · 10.1 Paket Kaggle · 11 *Random search* · 12 PSO · 13 Model akhir · 14 Evaluasi · 15 Uji signifikansi | ✔ | ditambahkan bertahap |

# A. Persiapan
Dua tahap ini dijalankan **di awal setiap sesi Colab**, sebelum tahap mana pun.

## Tahap 0.1 — Lingkungan
**Dasar:** Bab III (Google Colab, GPU T4); `REVISI_30OKT.md` R1, R2, R11.

- **Sel 1/2:** mengatur variabel determinisme (`CUBLAS_WORKSPACE_CONFIG`) dan menyenyapkan log, *mount* Google Drive, membuat struktur folder `skripsi_wondr_v4/`, lalu memasang *package* yang belum ada.
- **Sel 2/2:** meng-*import* *package*, memeriksa GPU (harus T4), lalu mencatat versi ke `log/versi_package.json`.

> Pada Langkah 2, sel ini ditambah `VERSI`, yaitu versi *package* yang dikunci beserta revisi *checkpoint* (R19).

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 1/2: variabel lingkungan, folder kerja, pemasangan package) ══
# Package: os, sys, subprocess, importlib, warnings, pathlib (bawaan Python); google.colab.drive
import os, sys, subprocess, importlib.util, warnings
from pathlib import Path
from google.colab import drive

# 1) Variabel lingkungan, diatur SEBELUM torch/transformers di-import (3.3–3.4)
_cuda_dulu = "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized()
_cublas_dulu = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
os.environ.update({
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",                                          # determinisme cuBLAS
    "TOKENIZERS_PARALLELISM": "false",                                             # senyap: tokenizer
    "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",   # senyap: transformers
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_DATASETS_DISABLE_PROGRESS_BARS": "1", # senyap: HF Hub, datasets
    "TQDM_DISABLE": "1",                                                           # senyap: semua tqdm
    "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3",                                    # jangan muat TensorFlow
})
warnings.filterwarnings("ignore")

# 2) Google Drive dan struktur folder (3.1); folder versi lama tidak disentuh
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount gagal: pilih akun yang menyimpan folder skripsi, centang SEMUA izin, ulangi sel.") from e
AKAR = Path("/content/drive/MyDrive/skripsi_wondr_v4")    # R2: folder v4; skripsi_wondr_v3 tidak disentuh
AKAR.mkdir(exist_ok=True)                         # gagal jika MyDrive belum ter-mount
for sub in ["data", "lib", "hasil", "anotasi", "bab4/tabel", "bab4/gambar", "log"]:
    (AKAR / sub).mkdir(parents=True, exist_ok=True)
(AKAR / "lib" / "__init__.py").touch()            # agar "from lib.wondr_teks import ..." jalan
os.chdir(AKAR)                                     # agar %%writefile cukup memakai lib/...
if str(AKAR) not in sys.path:
    sys.path.insert(0, str(AKAR))

# 3) Pasang hanya package yang belum ada (di-import di sel 2/2)
PAKET = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "transformers": "transformers",
         "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
         "openpyxl": "openpyxl", "pyarrow": "pyarrow"}            # nama modul -> nama pip
_dipasang = [pip for mod, pip in PAKET.items() if importlib.util.find_spec(mod) is None]
if _dipasang:
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_dipasang], capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + _r.stderr[-2000:])

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 2/2: import, GPU, dan log versi) ══
# Package: importlib, json, time, platform (bawaan Python); numpy, pandas, torch, transformers,
#          matplotlib (di-import); scikit-learn, scipy, openpyxl, pyarrow (versinya dicatat)
import importlib.metadata, json, time, platform
t0 = time.perf_counter()
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, transformers
from huggingface_hub.utils import logging as hf_logging

# 1) Matikan kebisingan yang tersisa setelah import
transformers.logging.set_verbosity_error()        # sembunyikan "Some weights ... newly initialized"
transformers.logging.disable_progress_bar()       # progress bar unduhan model
hf_logging.set_verbosity_error()                  # sembunyikan peringatan HF Hub "unauthenticated requests"
plt.rcParams.update({"savefig.dpi": 300, "savefig.bbox": "tight"})   # PNG 300 dpi (3.1)

# 2) GPU dan versi -> log/versi_package.json (acuan = sesi T4 pertama)
GPU_PENELITIAN = "T4"                              # R11: Bab III tahap 10; seluruh pelatihan di T4
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada"
mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if gpu != "tidak ada" else 0.0
sesi = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "gpu": gpu,
        "gpu_memori_gb": round(mem, 2), "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "package": {p: importlib.metadata.version(p) for p in [*PAKET.values(), "tokenizers", "huggingface-hub"]}}
log = AKAR / "log" / "versi_package.json"
lama = json.loads(log.read_text()) if log.exists() else {}
pertama = lama["pertama"] if GPU_PENELITIAN in lama.get("pertama", {}).get("gpu", "") else sesi
log.write_text(json.dumps({"pertama": pertama, "terbaru": sesi}, indent=2))
beda = [k for k in ("torch", "transformers", "tokenizers") if pertama["package"][k] != sesi["package"][k]]
beda += ["GPU"] if pertama["gpu"] != gpu else []

# 3) Ringkasan + cek (judul() dan cek() baru dibuat di Tahap 0.2)
csv, koma = AKAR / "data" / "review_wondr_bni_databaruu.csv", lambda x: f"{x:.1f}".replace(".", ",")   # R1
cublas_ok = not _cuda_dulu or _cublas_dulu == ":4096:8"
print(f"══════════ TAHAP 0.1 — LINGKUNGAN ══════════\nMasukan : data/{csv.name}\nKeluaran: log/{log.name}")
print(f"{'─' * 48}\nFolder kerja : {AKAR}\nPython       : {sesi['python']}\n"
      f"GPU          : {gpu} ({koma(mem)} GB) · CUDA {sesi['cuda']} · cuDNN {sesi['cudnn']}\n\n{'Package':<17}Versi")
print("\n".join(f"{k:<17}{v}{'  (baru dipasang)' * (k in _dipasang)}" for k, v in sesi["package"].items()))
print(f"{'─' * 48}\nCek otomatis:")
for nama, ok, aktual, target in [
        ("berkas data ada", csv.exists(), "ya" if csv.exists() else "tidak", "ya"),
        ("GPU", GPU_PENELITIAN in gpu, gpu, f"memuat {GPU_PENELITIAN}"),
        ("CUBLAS_WORKSPACE_CONFIG diatur sebelum CUDA aktif", cublas_ok, "ya" if cublas_ok else "tidak, restart runtime", "ya"),
        ("versi torch/transformers/tokenizers/GPU", not beda, "berubah: " + ", ".join(beda) if beda else "sama", "sama")]:
    print(f"  {'✅' if ok else '⚠️'} {nama} = {aktual} (target {target})")
print(f"Waktu: {koma(time.perf_counter() - t0)} detik")

## Tahap 0.2 — Konfigurasi dan fungsi bantu
**Dasar:** Bab III (Tabel 3.1 dan 3.2); `CLAUDE.md` Bagian 4; `REVISI_30OKT.md` R3; `REVISI_BRODY.md` R15; **`REVISI_12NOV.md` R17**; **anggaran 40** (keputusan pengguna 13-11-2026, menggantikan R27).

- **Sel 1/2** menulis `lib/bantu.py`, yang berisi format angka Indonesia, judul dan cek ringkasan, fungsi simpan tabel/gambar/JSON, serta `atur_seed()` (determinisme dengan TF32 mati).
- **Sel 2/2** membuat `PATH` (lokasi berkas), `CFG` (semua angka konfigurasi), dan `TARGET` (angka target setiap tahap).

***Seed*:** pembagian 42 · CL 42 · lembar anotasi 2026 · *fitness* 42 dan 43 · *random search* 1–5 · PSO 11–15 · model akhir 44–48 · *bootstrap* 42. Anggaran: **40 evaluasi per *run***, tanpa Tahap B. RS memakai 40 kandidat, dan PSO memakai 10 partikel × (1 + 3 iterasi).

In [ ]:
%%writefile lib/bantu.py
# lib/bantu.py -- fungsi bantu untuk semua subtahap (Tahap 0.2; KT-4 disetujui 21-10-2026).
# Isinya fungsi saja; PATH, CFG, dan TARGET dibuat di sel 2/2 Tahap 0.2.
import json, math, os, random, time
from pathlib import Path
import numpy as np

__all__ = ["fmt_id", "rel", "mulai", "judul", "garis", "cek", "selesai",
           "simpan_tabel", "simpan_gambar", "simpan_json", "baca_json", "atur_seed"]
_AKAR = Path(__file__).resolve().parents[1]          # folder kerja .../skripsi_wondr_v3
_STATUS = {"t0": None, "cek_mulai": False}
_LEBAR = 48                                           # lebar garis pemisah blok ringkasan


def _angka(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def fmt_id(x, d=None):
    """Format angka Indonesia: 7204 -> '7.204'; 0.836 -> '0,8360'. Selain angka -> str(x)."""
    if not _angka(x):
        return str(x)
    if d is None:
        d = 0 if isinstance(x, (int, np.integer)) else 4
    return f"{x:,.{d}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def _tampil(v, d=None):
    """Nilai untuk ringkasan: bilangan bulat tanpa desimal, pecahan d desimal, daftar dipisah ' · '."""
    if isinstance(v, np.ndarray):
        v = v.tolist()
    if isinstance(v, dict):
        return ", ".join(f"{k}: {_tampil(x, d)}" for k, x in v.items())
    if isinstance(v, (list, tuple)):
        return " · ".join(f"[{_tampil(x, d)}]" if isinstance(x, (list, tuple)) else _tampil(x, d) for x in v)
    return fmt_id(v) if isinstance(v, (int, np.integer)) else fmt_id(v, d)


def rel(p):
    """Path relatif terhadap folder kerja, untuk ditampilkan."""
    try:
        return str(Path(p).resolve().relative_to(_AKAR))
    except ValueError:
        return str(p)


def garis():
    print("─" * _LEBAR)


def mulai():
    """Dipanggil di awal sel: mulai menghitung waktu."""
    _STATUS["t0"] = time.perf_counter()


def judul(no, nama, masukan=None, keluaran=None):
    """Kepala blok ringkasan; masukan/keluaran berupa str atau list of str."""
    _STATUS["cek_mulai"] = False
    print(f"{'═' * 10} TAHAP {no} — {nama.upper()} {'═' * 10}")
    for label, isi in (("Masukan ", masukan), ("Keluaran", keluaran)):
        if isi:
            isi = [isi] if isinstance(isi, str) else list(isi)
            print(f"{label}: " + f"\n{' ' * 10}".join(isi))
    garis()


def _sama(a, t, tol):
    a = a.tolist() if isinstance(a, np.ndarray) else a
    t = t.tolist() if isinstance(t, np.ndarray) else t
    if _angka(a) and _angka(t):
        return abs(float(a) - float(t)) <= tol
    if isinstance(a, dict) and isinstance(t, dict):
        return a.keys() == t.keys() and all(_sama(a[k], t[k], tol) for k in t)
    if isinstance(a, (list, tuple)) and isinstance(t, (list, tuple)):
        return len(a) == len(t) and all(_sama(x, y, tol) for x, y in zip(a, t))
    return bool(a == t)


def cek(nama, aktual, target, toleransi=0, cara="=", d=None):
    """Cetak ✅/⚠️ beserta nilai aktual dan target, tanpa assert. cara: '=', '>', '>=', '<', '<='."""
    if not _STATUS["cek_mulai"]:
        garis()
        print("Cek otomatis:")
        _STATUS["cek_mulai"] = True
    if d is None and toleransi > 0:
        d = max(0, round(-math.log10(2 * toleransi)))  # 0,05 -> 1 desimal; 5e-5 -> 4 desimal
    if cara == "=":
        ok = _sama(aktual, target, toleransi)
    else:
        a, t = float(aktual), float(target)
        ok = {">": a > t, ">=": a >= t, "<": a < t, "<=": a <= t}[cara]
    tanda = "" if cara == "=" else cara + " "
    print(f"  {'✅' if ok else '⚠️'} {nama} = {_tampil(aktual, d)} (target {tanda}{_tampil(target, d)})")
    return bool(ok)


def selesai():
    """Baris terakhir blok ringkasan: lama sel berjalan sejak mulai()."""
    dt = time.perf_counter() - _STATUS["t0"] if _STATUS["t0"] else 0.0
    teks = (f"{fmt_id(dt, 1)} detik" if dt < 120 else
            f"{fmt_id(dt / 60, 1)} menit" if dt < 7200 else f"{fmt_id(dt / 3600, 2)} jam")
    print(f"Waktu: {teks}")


def simpan_tabel(df, nama, index=False):
    """CSV untuk Bab IV (angka mentah) -> bab4/tabel/<nama>.csv; mengembalikan path-nya."""
    p = _AKAR / "bab4" / "tabel" / f"{nama}.csv"
    df.to_csv(p, index=index)
    return p


def simpan_gambar(fig, nama):
    """PNG 300 dpi -> bab4/gambar/<nama>.png, lalu ditampilkan satu kali."""
    import matplotlib.pyplot as plt
    p = _AKAR / "bab4" / "gambar" / f"{nama}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return p


def _ke_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, (np.ndarray, Path)):
        return o.tolist() if isinstance(o, np.ndarray) else str(o)
    raise TypeError(f"tidak bisa disimpan ke JSON: {type(o).__name__}")


def simpan_json(obj, path):
    """Tulis JSON secara atomik (berkas .tmp lalu diganti), aman bila Colab terputus."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_ke_json), encoding="utf-8")
    os.replace(tmp, path)
    return path


def baca_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atur_seed(s):
    """Satu seed untuk semua sumber keacakan pelatihan (3.4), ditambah KT-2 dan KT-3 (21-10-2026)."""
    import torch, transformers
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    transformers.set_seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)         # KT-2: operasi GPU tak-deterministik -> error
    torch.backends.cuda.matmul.allow_tf32 = False    # KT-3: FP32 penuh; API lama ikut mengatur API baru,
    torch.backends.cudnn.allow_tf32 = False          #       API baru saja tidak mematikan TF32 konv/RNN cuDNN


def uji_bantu(s):
    """Uji Tahap 0.2: atur_seed(s) dua kali -> angka acak sama; status mode deterministik dan TF32."""
    import torch
    atur_seed(s); a = np.random.rand(3), torch.rand(3)
    atur_seed(s); b = np.random.rand(3), torch.rand(3)
    return {"ulang_sama": all(np.array_equal(x, y) for x, y in zip(a, b)),
            "deterministik": torch.are_deterministic_algorithms_enabled(),
            "tf32": [torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32]}

In [ ]:
# ══ Tahap 0.2 — Konfigurasi dan fungsi bantu (sel 2/2: PATH, CFG, TARGET) ══
# Package: importlib (bawaan Python); lib.bantu (fungsi bantu dari sel 1/2)
import importlib, lib.bantu
importlib.reload(lib.bantu)                         # pakai isi lib/bantu.py yang terbaru
from lib.bantu import *
mulai()

# 1) Lokasi folder dan berkas (3.1)
PATH = {"akar": AKAR, **{k: AKAR / k for k in ("data", "lib", "hasil", "anotasi", "log")},
        "tabel": AKAR / "bab4/tabel", "gambar": AKAR / "bab4/gambar", "csv": AKAR / "data/review_wondr_bni_databaruu.csv"}

# 2) Konfigurasi tetap (CLAUDE.md Bagian 4; REVISI_30OKT R3, R11; REVISI_BRODY R15; REVISI_12NOV R17; anggaran 40) -- jangan diubah
CFG = {
    "gpu": "T4",                                                    # R11; Bab III tahap 10
    "checkpoint": "indobenchmark/indobert-base-p1",                 # Bab III tahap 7; Wilie dkk. (2020)
    "tau": 4, "kelas": {0: "Non-Positif", 1: "Positif"},            # pers. (2.1)-(2.2): rating >= 4 -> 1
    "brody_ambang": 10,                                             # Brody & Diakopoulos (2011); Bab II 2.2.4
    "proporsi": {"latih": 0.70, "validasi": 0.15, "uji": 0.15},     # Bab III tahap 6
    "seed_bagi": 42, "kelompok_panjang": ("1-2", "3-10", ">10"),    # Bab III tahap 6
    "q": 100, "b": 32,                                              # pers. (2.9): L_max dari data latih saja
    "cl_lipatan": 5, "cl_seed_lipatan": 42, "cl_seed_latih": 42,    # Bab III tahap 8; Keputusan teknis
    "seed_anotasi": 2026,                                           # Keputusan: urutan lembar anotasi (tahap 9)
    "B": 16, "E": 4, "p": 2, "v": 1.0,                              # Tabel 3.1: batch, epoch, patience, norma
    "adam_betas": (0.9, 0.999), "adam_eps": 1e-8,                   # pers. (2.18); bawaan PyTorch
    "default": {"lr": 2e-5, "wr": 0.1, "wd": 0.01},                 # Tabel 3.1; Devlin dkk.; Sun dkk.
    "ruang": {"lr": (1e-5, 5e-5), "wr": (0.0, 0.2), "wd": (0.0, 0.3)},   # Tabel 3.1; pers. (2.29); lr skala log
    "seed_fitness": (42, 43),                                       # Bab III tahap 10; pers. (2.23), (2.25)-(2.26)
    "rs_M": 40, "rs_seeds": tuple(range(1, 6)),                     # Bab III tahap 11; R17: 5 run, seed 1-5; 40 kandidat
    "pso": {"s": 10, "G_max": 3, "omega": 0.729, "c1": 1.49445, "c2": 1.49445, "V_max": 1.0},   # Tabel 3.2; 10 x (1 + 3) = 40
    "pso_seeds": tuple(range(11, 16)),                              # Bab III tahap 12; R17: 5 run, seed 11-15
    "seed_akhir": (44, 45, 46, 47, 48),                             # Bab III tahap 13; R3, R6
    "R": 10_000, "seed_bootstrap": 42,                              # Bab III tahap 15; pers. (2.43)-(2.45)
    "alpha": 0.05, "m_uji": 3,                                      # pers. (2.48)-(2.49): alpha' = 0,05/3
}

# 3) Angka target (REVISI_30OKT Bagian 3, REVISI_BRODY Bagian 5, REVISI_12NOV Bagian 4; nomor tahap mengikuti Bab III)
TARGET = {
    "1.1": {"baris": 9672, "kolom": ["nama", "rating", "ulasan", "tanggal", "likes", "balasan_dev", "versi"],
            "tanggal": ("2026-07-19", "2026-10-04"), "versi": ["1.6.0"],
            "per_bulan": {"2026-07": 1734, "2026-08": 3857, "2026-09": 3525, "2026-10": 556}},
    "1.2": {"kosong": {"balasan_dev": 4474}, "dibalas_pct": {"1-4": 99.6, "5": 30.8}, "likes_nol_pct": 95.1,
            "spearman": -0.2404, "r3_panjang": (315, 479), "median_kata": [15, 15, 11, 2, 2], "emoji_simbol": 549,
            "huruf_ulang": (326, 3.4), "baca_ulang": 797, "duplikat_pct": 36.7, "label_ganda": (42, 2721),
            "nama_unik": 9573},
    "2": {"baris": 9672, "kolom": ["review_id", "ulasan", "rating", "tanggal", "versi"]},
    "3": {"per_rating": {1: 1713, 2: 372, 3: 479, 4: 659, 5: 6449}, "positif": (7108, 73.49),
          "non_positif": (2564, 26.51)},
    "4": {"contoh_bab2": ["bagus bgt! tp kadang2 error, nunggu lama", "mantap"],
          "berubah": [66, 549, 2001, 326, 797, 547], "kosong": 92, "kosong_rating": {5: 85, 4: 6, 1: 1}, "hasil": 9580,
          "kamus": {"kelompok_memanjang": 176, "buang_1_huruf": 0, "buang_frekuensi": 88,
                    "buang_kanonik_memanjang": 0, "dipakai": 88},
          "memanjang": {"total": 381, "kamus": 266, "cadangan": 115, "bentuk": (260, 158, 102)}},   # R13-R15
    "5": {"kunci": 5684, "kunci_seri": 11, "ulasan_seri": 24, "dokumen": 5673, "dibuang": (3907, 3784, 99, 24),
          "positif_pct": (73.2, 57.0), "gabungan": (49, 0)},        # gabungan: kunci terdampak, memuat angka (R9)
    "6.1": {"grup": 5563, "latih": (3969, 1708, 2261), "validasi": (852, 366, 486), "uji": (852, 367, 485)},
    "6.2": {"bocor": 0, "non_positif_pct": (43.0, 43.0, 43.1), "p_label": 0.9987,
            "p_panjang": 0.9561, "panjang_pct": {"latih": (13.5, 47.7, 38.8), "validasi": (14.4, 46.8, 38.7),
                                                 "uji": (13.3, 48.0, 38.7)},
            "dampak": {"ulasan_latih": 6798, "kamus_latih": {"kelompok_memanjang": 131, "buang_1_huruf": 0,
                       "buang_frekuensi": 64, "buang_kanonik_memanjang": 0, "dipakai": 67},
                       "berbeda": {"latih": 4, "validasi": 9, "uji": 4},
                       "berbeda_id": {"validasi": [1868, 2732, 4911, 6773, 7582, 7593, 7609, 8671, 9371],
                                      "uji": [3668, 5741, 7605, 7928]}}},   # R16: cek dampak kamus Brody
    "7": {"L_max": 160, "terpotong": {"latih": 0, "validasi": 0, "uji": 0},
          "terpotong_id": []},                                      # pers. (2.9): maks latih 141 -> 160 (keputusan 31-10)
    "8.1": {"T": 996, "T_w": 99},                                   # n = 3.969, rho = 0,1
    "8.3": {"t0": 0.6167, "t1": 0.6667, "C": [[2, 1], [1, 2]], "kandidat": [3, 6]},    # contoh Tabel 2.9
    "9.2": {"kappa_contoh": 0.70, "toleransi": 1e-12},              # contoh Bab II 2.3.5
    "10": {"T": 996, "T_w": 99}, "11": {"run": 5, "kandidat": 40, "pelatihan": 80},     # 11: kandidat/pelatihan per run
    "12.1": {"iter1": (5.6237, 3.6355), "iter2": (2.6718, 3.3698), "gbest_iter2": 9.8923,
             "toleransi": 5e-4, "run": 5, "evaluasi": 40},          # contoh Tabel 2.17; evaluasi per run
    "13": {"pelatihan": 15, "toleransi": 1e-4}, "14.1": {"n_uji": 852},
    "15": {"f1_contoh": 0.6703, "persentil": (0.8333, 99.1667), "toleransi": 1e-12},   # contoh Tabel 2.19
}

# 4) Uji singkat fungsi bantu, lalu ringkasan
u = lib.bantu.uji_bantu(CFG["seed_fitness"][0])
rentang = lambda s: f"{s[0]}–{s[-1]}" if list(s) == list(range(s[0], s[-1] + 1)) else ", ".join(map(str, s))
judul("0.2", "Konfigurasi dan fungsi bantu", masukan="lib/bantu.py", keluaran="PATH, CFG, TARGET (di memori)")
print(f"CFG {len(CFG)} entri · TARGET {len(TARGET)} subtahap · PATH {len(PATH)} lokasi · "
      f"fungsi bantu {len(lib.bantu.__all__)}")
print(f"seed: fitness {rentang(CFG['seed_fitness'])} · RS {rentang(CFG['rs_seeds'])} · PSO {rentang(CFG['pso_seeds'])} · "
      f"model akhir {rentang(CFG['seed_akhir'])} · GPU {CFG['gpu']}")
cek("fmt_id(7204) · fmt_id(0.836)", [fmt_id(7204), fmt_id(0.836)], ["7.204", "0,8360"])
cek("atur_seed() dua kali, angka acak sama", "ya" if u["ulang_sama"] else "tidak", "ya")
cek("mode deterministik PyTorch (KT-2)", "aktif" if u["deterministik"] else "mati", "aktif")
cek("TF32 matmul · cuDNN (KT-3)", ["nyala" if x else "mati" for x in u["tf32"]], ["mati", "mati"])
S3 = [set(CFG[k]) for k in ("seed_fitness", "rs_seeds", "pso_seeds", "seed_akhir")]
cek("run RS · PSO (R17) · seed akhir (R3) · tidak bertumpuk · kunci lama terhapus",
    [len(CFG["rs_seeds"]), len(CFG["pso_seeds"]), len(CFG["seed_akhir"]),
     "ya" if sum(map(len, S3)) == len(set().union(*S3)) else "tidak",
     "ya" if not ({"rs_seed", "kembaran_dekat"} & set(CFG) or "seed" in CFG["pso"]) else "tidak"],
    [TARGET["11"]["run"], TARGET["12.1"]["run"], TARGET["13"]["pelatihan"] // 3, "ya", "ya"])   # 15 pelatihan / 3 skema
cek("evaluasi per run: RS · PSO (anggaran 40, keputusan 13-11-2026)",
    [CFG["rs_M"], CFG["pso"]["s"] * (1 + CFG["pso"]["G_max"])], [TARGET["11"]["kandidat"], TARGET["12.1"]["evaluasi"]])
selesai()

# B. Data (Tahap 1–7)
**Sudah final; jangan dijalankan ulang.** Data mentah diolah menjadi data latih, validasi, dan uji. Mulai Tahap 6.2, data uji dikunci.

## Tahap 1.1 — Muat data mentah
**Dasar:** Bab III tahap 1.

Membaca `data/review_wondr_bni_databaruu.csv` apa adanya (9.672 ulasan, 19 Juli–4 Oktober 2026, versi aplikasi 1.6.0), lalu menyimpannya ke `hasil/tahap01_mentah.parquet`.

In [ ]:
# ══ Tahap 1.1 — Muat data mentah ══
# Package: pandas (baca .csv, olah tabel), pyarrow (mesin simpan .parquet)
import pandas as pd
mulai()
T = TARGET["1.1"]
BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
         "Agustus", "September", "Oktober", "November", "Desember"]
tgl_id = lambda t: f"{t.day} {BULAN[t.month - 1]} {t.year}"       # 2026-07-19 -> "19 Juli 2026"

# 1) Baca data mentah (.csv, R1); versi dibaca sebagai teks agar "1.6.0" tidak pernah ditafsirkan sebagai angka
mentah = pd.read_csv(PATH["csv"], dtype={"versi": str})

# 2) Buang kolom indeks bawaan (Unnamed*), bila ada; isi tidak diubah, review_id dibuat di Tahap 2
buang = [c for c in mentah.columns if str(c).startswith("Unnamed")]
df = mentah.drop(columns=buang)

# 3) Simpan, lalu pastikan berkas yang dibaca ulang identik
keluar = PATH["hasil"] / "tahap01_mentah.parquet"
df.to_parquet(keluar, index=False)
identik = df.equals(pd.read_parquet(keluar))

# 4) Ringkasan + cek
tgl = pd.to_datetime(df["tanggal"])                                # hanya untuk ringkasan; kolom asli tetap
per_bulan = {k: int(v) for k, v in tgl.dt.strftime("%Y-%m").value_counts().sort_index().items()}
versi = sorted(df["versi"].astype(str).unique())
judul("1.1", "Muat data mentah", masukan=f"{rel(PATH['csv'])} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"Kolom dibuang  : {', '.join(map(str, buang)) or '-'}")
print(f"Rentang tanggal: {tgl_id(tgl.min())} – {tgl_id(tgl.max())}")
print(f"Versi aplikasi : {', '.join(versi)}\n\n{'Bulan':<16}{'Ulasan':>7}")
for k, v in per_bulan.items():
    print(f"{BULAN[int(k[5:]) - 1] + ' ' + k[:4]:<16}{fmt_id(v):>7}")
print(f"{'Total':<16}{fmt_id(sum(per_bulan.values())):>7}")
cek("jumlah baris", len(df), T["baris"])
cek("nama dan urutan kolom", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(map(str, df.columns)), "sesuai")
cek("rentang tanggal", (str(tgl.min().date()), str(tgl.max().date())), T["tanggal"])
cek("nilai unik versi", versi, T["versi"])
cek("ulasan per bulan", per_bulan, T["per_bulan"])
cek("parquet identik dengan isi .csv", "ya" if identik else "tidak", "ya")
selesai()

## Tahap 1.2 — Kode pengolah teks dan audit kualitas data *(audit: program saja)*
**Dasar:** Bab III tahap 1–6; Bab II 2.2; Brody dan Diakopoulos (2011).

- **Sel 1/2** menulis `lib/wondr_teks.py`. Isinya: pola teks, enam langkah *pre-processing*, kamus pemanjangan kata Brody, deduplikasi, kunci grup, dan pembagian data. Kode ini dipakai Tahap 1.2–6.
- **Sel 2/2** mengaudit data mentah tanpa mengubahnya: nilai kosong, *likes*, panjang ulasan, emoji, pengulangan huruf, duplikat, dan teks berlabel ganda. Hasilnya disimpan ke `bab4/tabel/tahap01_audit.csv`.

In [ ]:
%%writefile lib/wondr_teks.py
# lib/wondr_teks.py -- kode acuan Bagian 7 CLAUDE.md dan Bagian 3 REVISI_BRODY.md;
# dipakai Tahap 1.2, 4, 5, 6, dan 6.2 (cek dampak kamus).
import re, unicodedata
from collections import Counter, defaultdict
import numpy as np, pandas as pd

# -- Pola teks (Tahap 1.2 dan Tahap 4 memakai pola yang sama) --
EMO = re.compile("[\U0001F000-\U0001FAFF\U0001F1E6-\U0001F1FF\u2600-\u27BF\u2B00-\u2BFF"
                 "\u2300-\u23FF\u2190-\u21FF\u2900-\u297F\u3030\u303D\u3297\u3299"
                 "\u00A9\u00AE\u203C\u2049\u2122\u2139\u24C2\u25AA-\u25FE\uFE0F\u200D"
                 "\U000E0020-\U000E007F]")
SIMBOL = re.compile("[\u2600-\u27BF\u2B00-\u2BFF\uFE0F\u200D\U0001F3FB-\U0001F3FF]")
HURUF = re.compile(r"([^\W\d_])\1{2,}")      # huruf yang berulang >= 3 kali (hanya huruf)
BACA = re.compile(r"[!?.,;:]{2,}")           # tanda baca berurutan >= 2

# -- Tahap 4 langkah 4: normalisasi pemanjangan kata (Brody & Diakopoulos, 2011), lalu cadangan >=3 -> 2 --
KATA = re.compile(r"[^\W\d_]+")    # kata = deretan huruf; angka dan tanda baca tidak termasuk kata

def ringkas(w):                    # semua huruf berurutan yang sama -> satu (hanya kunci kelompok)
    return re.sub(r"([^\W\d_])\1+", r"\1", w)

def kamus_brody(teks, ambang=10):
    """Kamus {kunci ringkas: bentuk kanonik} dari frekuensi kata seluruh teks; label tidak dipakai."""
    frek = Counter(w for t in teks for w in KATA.findall(t))
    grup = defaultdict(dict)
    for w, n in frek.items():
        grup[ringkas(w)][w] = n
    kamus, info = {}, {"kelompok_memanjang": 0, "buang_1_huruf": 0, "buang_frekuensi": 0, "buang_kanonik_memanjang": 0}
    for k in sorted(grup):
        g = grup[k]
        if not any(HURUF.search(w) for w in g):                 # kelompok tanpa bentuk memanjang
            continue
        info["kelompok_memanjang"] += 1
        kan, n = sorted(g.items(), key=lambda x: (-x[1], len(x[0]), x[0]))[0]   # tersering; seri: terpendek, abjad
        if len(kan) == 1:
            info["buang_1_huruf"] += 1
        elif n < ambang:
            info["buang_frekuensi"] += 1
        elif HURUF.search(kan):
            info["buang_kanonik_memanjang"] += 1
        else:
            kamus[k] = kan
    return kamus, info

def pemanjangan(s, kamus):
    """Kata berhuruf berulang >= 3: bentuk kanonik bila kelompoknya ada di kamus; selain itu >= 3 -> 2."""
    def ganti(m):
        w = m.group()
        return kamus.get(ringkas(w), HURUF.sub(r"\1\1", w)) if HURUF.search(w) else w
    return KATA.sub(ganti, s)

def langkah(kamus):  # (nama, fungsi) -- urutan Tahap 4, jangan diubah
    return [
        ("1. Normalisasi Unicode NFKC", lambda s: unicodedata.normalize("NFKC", s)),
        ("2. Hapus emoji dan simbol", lambda s: SIMBOL.sub(" ", EMO.sub(" ", s))),
        ("3. Case folding", lambda s: s.lower()),
        ("4. Pemanjangan kata (Brody, >=3->2)", lambda s: pemanjangan(s, kamus)),
        ("5. Tanda baca berulang -> 1", lambda s: BACA.sub(lambda m: m.group(0)[0], s)),
        ("6. Rapikan spasi", lambda s: re.sub(r"\s+", " ", s).strip()),
    ]

def sebelum_kamus(s):              # langkah 1-3 saja: teks yang dihitung untuk membentuk kamus
    for _, f in langkah({})[:3]:
        s = f(s)
    return s

def bersihkan(s, kamus):
    s = str(s)
    for _, f in langkah(kamus):
        s = f(s)
    return s

# -- Tahap 5: deduplikasi --
def kunci_dedup(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s))   # sisakan a-z, 0-9, spasi
    s = re.sub(r"(.)\1+", r"\1", s)          # karakter berurutan yang sama -> satu
    return re.sub(r"\s+", " ", s).strip()

def deduplikasi(d):
    d = d.copy()
    d["dedup_key"] = d["text_clean"].map(kunci_dedup)
    d = d[d["dedup_key"] != ""]
    suara = (d.groupby(["dedup_key", "label"]).size().unstack(fill_value=0)
               .reindex(columns=[0, 1], fill_value=0))
    suara.columns = ["n_np", "n_p"]
    seri = suara["n_np"] == suara["n_p"]
    suara["mayor"] = (suara["n_p"] > suara["n_np"]).astype(int)
    sah = suara[~seri]
    x = d[d["dedup_key"].isin(sah.index)].copy()
    x = x[x["label"] == x["dedup_key"].map(sah["mayor"])]          # anggota berlabel mayoritas
    frek = x.groupby(["dedup_key", "text_clean"]).size().rename("_f").reset_index()
    x = x.merge(frek, on=["dedup_key", "text_clean"])
    wakil = (x.sort_values(["dedup_key", "_f", "review_id"], ascending=[True, False, True])
               .groupby("dedup_key").head(1)["review_id"])           # teks tersering, lalu id terkecil
    hasil = d[d["review_id"].isin(set(wakil))].copy()
    hasil["label"] = hasil["dedup_key"].map(sah["mayor"]).astype(int)
    return hasil.sort_values("review_id").reset_index(drop=True), suara, seri

# -- Tahap 6: pembagian data berbasis grup --
KATA_FUNGSI = {"dan", "yang", "nya", "ini", "itu", "ya", "sih", "deh", "juga", "aja", "saja", "nih",
               "kok", "dong", "lah", "di", "ke", "dari", "untuk", "pada", "dengan", "yg", "jg", "utk", "dgn"}

def kunci_grup(s):                      # hanya untuk pengelompokan; teks tidak diubah
    k = set(str(s).split())
    inti = k - KATA_FUNGSI
    return " ".join(sorted(inti)) if inti else " ".join(sorted(k))

def kelompok_panjang(n_kata):
    return "1-2" if n_kata <= 2 else ("3-10" if n_kata <= 10 else ">10")

PROPORSI = {"latih": 0.70, "validasi": 0.15, "uji": 0.15}   # urutan kunci menentukan pemecah seri

def bagi_data(dd, seed=42):
    dd = dd.copy()
    dd["group_key"] = dd["text_clean"].map(kunci_grup)
    dd["n_kata"] = dd["text_clean"].str.split().str.len()
    label_id_kecil = dd.sort_values("review_id").groupby("group_key")["label"].first()
    g = (dd.groupby("group_key")
           .agg(n=("review_id", "size"), n_p=("label", "sum"), kata=("n_kata", "median"))
           .reset_index())
    mayor = np.where(g.n_p * 2 > g.n, 1, np.where(g.n_p * 2 < g.n, 0, -1))
    g["label_grup"] = np.where(mayor == -1, g.group_key.map(label_id_kecil).to_numpy(), mayor).astype(int)
    g["kelompok_panjang"] = g["kata"].map(kelompok_panjang)
    rng = np.random.default_rng(seed)                     # satu generator untuk semua strata
    alokasi = {}
    for _, sub in g.groupby(["label_grup", "kelompok_panjang"], sort=True):
        sub = sub.iloc[rng.permutation(len(sub))]
        total = int(sub.n.sum())
        target = {b: p * total for b, p in PROPORSI.items()}
        terisi = {b: 0 for b in PROPORSI}
        for kunci, n in zip(sub.group_key, sub.n):
            b = min(terisi, key=lambda b: terisi[b] / target[b])   # keterisian terkecil
            alokasi[kunci] = b
            terisi[b] += int(n)
    dd["split"] = dd["group_key"].map(alokasi)
    dd["kelompok_panjang"] = dd["n_kata"].map(kelompok_panjang)
    return dd, g

In [ ]:
# ══ Tahap 1.2 — Audit kualitas data (sel 2/2; program saja, data tidak diubah) ══
# Package: json, importlib (bawaan Python); pandas; scipy.stats.spearmanr; lib.wondr_teks (pola teks)
import json, importlib
import pandas as pd
from scipy.stats import spearmanr
import lib.wondr_teks as wt
from lib.bantu import _tampil                              # format angka Indonesia untuk tabel
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["1.2"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap01_mentah.parquet"
df = pd.read_parquet(masuk)
u, r = df["ulasan"].astype(str), df["rating"]
n_kata, teks = u.str.split().str.len(), u.str.lower().str.strip()    # kata = dipisah spasi

# 2) Pemeriksaan (data hanya dibaca); label sementara = rating >= tau, hanya untuk audit
dibalas, huruf = df["balasan_dev"].notna(), u.str.lower().map(lambda s: bool(wt.HURUF.search(s)))
n_lbl = (r >= CFG["tau"]).astype(int).groupby(teks).nunique()
med = [float(n_kata[r == k].median()) for k in range(1, 6)]
emoji = int(u.map(lambda s: bool(wt.EMO.search(s) or wt.SIMBOL.search(s))).sum())
AUDIT = [  # (temuan, nilai aktual, kunci TARGET, toleransi, tahap pemakai)
    ("Nilai kosong per kolom", {k: int(v) for k, v in df.isna().sum().items() if v}, "kosong", 0, "2"),
    ("Dibalas pengembang (%), rating 1–4 · 5",
     {"1-4": dibalas[r <= 4].mean() * 100, "5": dibalas[r == 5].mean() * 100}, "dibalas_pct", 0.05, "2"),
    ("Likes bernilai nol (%)", (df["likes"] == 0).mean() * 100, "likes_nol_pct", 0.05, "2"),
    ("Spearman likes–rating", spearmanr(df["likes"], r).statistic, "spearman", 5e-5, "2"),
    ("Nama unik", int(df["nama"].nunique()), "nama_unik", 0, "2"),
    ("Rating 3 >= 8 kata (jumlah · total)", (int(((r == 3) & (n_kata >= 8)).sum()), int((r == 3).sum())),
     "r3_panjang", 0, "3"),
    ("Median kata per rating 1–5", [int(m) if m == int(m) else m for m in med], "median_kata", 0, "6.1"),
    ("Baris berisi emoji atau simbol", emoji, "emoji_simbol", 0, "4"),
    ("Huruf berulang >= 3 (baris · %)", (int(huruf.sum()), huruf.mean() * 100), "huruf_ulang", 0.05, "4"),
    ("Tanda baca berulang (baris)", int(u.map(lambda s: bool(wt.BACA.search(s))).sum()), "baca_ulang", 0, "4"),
    ("Duplikat persis (% baris)", df["ulasan"].duplicated().mean() * 100, "duplikat_pct", 0.05, "5"),
    ("Teks berlabel ganda (teks · baris)", (int((n_lbl == 2).sum()), int(teks.isin(n_lbl[n_lbl == 2].index).sum())),
     "label_ganda", 0, "5"),
]

# 3) Simpan tabel (angka mentah, JSON), tampilkan temuan -> angka -> tahap pemakai, lalu cek
keluar = simpan_tabel(pd.DataFrame([{"temuan": t, "nilai": json.dumps(v), "target": json.dumps(T[k]), "tahap": th}
                                    for t, v, k, _, th in AUDIT]), "tahap01_audit")
judul("1.2", "Audit kualitas data", masukan=[f"{rel(masuk)} ({fmt_id(len(df))} baris)", "lib/wondr_teks.py"],
      keluaran=f"{rel(keluar)} ({len(AUDIT)} baris)")
print(f"{'Temuan':<40}{'Angka':<28}Tahap")
for t, v, _, tol, th in AUDIT:
    print(f"{t:<40}{_tampil(v, 1 if tol == 0.05 else 4):<28}{th}")
cek("jumlah baris tetap", len(df), TARGET["1.1"]["baris"])
for t, v, k, tol, _ in AUDIT:
    cek(t, v, T[k], toleransi=tol)
selesai()

## Tahap 2 — Seleksi atribut
**Dasar:** Bab III tahap 2; Bab II 2.4.2.

Membuat `review_id` (1–9.672), mempertahankan `review_id, ulasan, rating, tanggal, versi`, dan membuang `nama`, `balasan_dev`, serta `likes`. Keluaran: `hasil/tahap02_atribut.parquet`.

In [ ]:
# ══ Tahap 2 — Seleksi atribut ══
# Package: pandas (olah tabel), pyarrow (mesin baca/simpan .parquet)
import pandas as pd
mulai()
T = TARGET["2"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap01_mentah.parquet"
mentah = pd.read_parquet(masuk)

# 2) Nomor urut sebagai identitas, lalu pilih atribut (ringkasan kalimat Bab III tahap 2)
ATRIBUT = [  # (atribut, peran, alasan)
    ("review_id", "dibuat", "nomor urut sesuai urutan baris data mentah, sebagai identitas"),
    ("ulasan", "dipakai", "teks ulasan: satu-satunya masukan model"),
    ("rating", "dipakai", "sumber label awal (Tahap 3), bukan fitur"),
    ("tanggal", "disimpan", "informasi pendukung (subbab 2.4.2)"),
    ("versi", "disimpan", "informasi pendukung (subbab 2.4.2)"),
    ("nama", "dibuang", "nama pengguna; dihapus untuk menjaga privasi"),
    ("balasan_dev", "dibuang", "ditulis setelah ulasan; isinya dapat membocorkan label"),
    ("likes", "dibuang", "tidak tersedia pada saat prediksi"),
]
pakai = [a for a, peran, _ in ATRIBUT if peran != "dibuang"]
df = mentah.assign(review_id=range(1, len(mentah) + 1))[pakai]

# 3) Simpan
keluar = PATH["hasil"] / "tahap02_atribut.parquet"
df.to_parquet(keluar, index=False)

# 4) Ringkasan + cek
judul("2", "Seleksi atribut", masukan=f"{rel(masuk)} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"{'Atribut':<13}{'Peran':<10}Alasan")
for a, peran, alasan in ATRIBUT:
    print(f"{a:<13}{peran:<10}{alasan}")
tercatat = {a for a, _, _ in ATRIBUT} - {"review_id"} == set(mentah.columns)
ya = lambda b: "ya" if b else "tidak"
cek("jumlah baris", len(df), T["baris"])
cek("kolom dan urutannya", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(df.columns), "sesuai")
cek("semua kolom data mentah tercatat di tabel", ya(tercatat), "ya")
cek("review_id = 1, 2, ..., n (unik, urut)", ya(df["review_id"].tolist() == list(range(1, len(df) + 1))), "ya")
cek("isi ulasan, rating, tanggal, versi tidak berubah", ya(df[pakai[1:]].equals(mentah[pakai[1:]])), "ya")
selesai()

## Tahap 3 — Pelabelan awal (*top-2-box*)
**Dasar:** Bab III tahap 3; pers. (2.1)–(2.2), τ = 4.

`label` = 1 (Positif) bila *rating* ≥ 4, dan 0 (Non-Positif) bila *rating* ≤ 3. Keluaran: `hasil/tahap03_label.parquet` dan `bab4/gambar/tahap03_distribusi_rating.png`.

In [ ]:
# ══ Tahap 3 — Pelabelan awal (top-2-box) ══
# Package: pandas (olah tabel), matplotlib (grafik), pyarrow (mesin baca/simpan .parquet)
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
mulai()
T = TARGET["3"]
WARNA = {1: "#2a78d6", 0: "#eb6834"}          # slot kategori 1-2 palet tervalidasi (CVD aman); tetap per kelas

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap02_atribut.parquet"
df = pd.read_parquet(masuk)

# 2) Label awal pers. (2.1)-(2.2): rating >= tau -> 1 (Positif), selain itu 0 (Non-Positif); tidak ada baris dihapus
df["label"] = (df["rating"] >= CFG["tau"]).astype(int)
keluar = PATH["hasil"] / "tahap03_label.parquet"
df.to_parquet(keluar, index=False)

# 3) Gambar: jumlah ulasan per rating, diwarnai per kelas
per = df.groupby("rating").agg(n=("review_id", "size"), kelas=("label", "first"))
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(per.index, per.n, width=0.6, color=[WARNA[k] for k in per.kelas], zorder=2)
for r, n in per.n.items():
    ax.text(r, n + 60, fmt_id(int(n)), ha="center", va="bottom", fontsize=9, color="#52514e")
ax.set(xlabel="Rating", ylabel="Jumlah ulasan", xticks=per.index, ylim=(0, per.n.max() * 1.12))
ax.yaxis.set_major_formatter(lambda y, _: fmt_id(int(y)))
ax.grid(axis="y", color="#e4e3df", linewidth=0.8, zorder=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(colors="#52514e", length=0)
ax.legend(handles=[Patch(color=WARNA[k], label=CFG["kelas"][k]) for k in (1, 0)], frameon=False, loc="upper left")
gambar = simpan_gambar(fig, "tahap03_distribusi_rating")

# 4) Ringkasan + cek
kelas = {CFG["kelas"][k]: (int((df.label == k).sum()), (df.label == k).mean() * 100) for k in (1, 0)}
judul("3", "Pelabelan awal (top-2-box)", masukan=f"{rel(masuk)} ({fmt_id(len(df))} baris)",
      keluaran=[f"{rel(keluar)} ({fmt_id(len(df))} baris, kolom baru: label)", rel(gambar)])
print(f"{'Rating':<8}{'Jumlah':>8}  Kelas")
for r, (n, k) in per.iterrows():
    print(f"{r:<8}{fmt_id(int(n)):>8}  {CFG['kelas'][k]}")
for nama, (n, pct) in kelas.items():
    print(f"Total {nama:<12}{fmt_id(n):>7}  ({fmt_id(pct, 2)}%)")
cek("jumlah baris tetap (tidak ada yang dihapus)", len(df), TARGET["2"]["baris"])
cek("jumlah per rating", {int(r): int(n) for r, n in per.n.items()}, T["per_rating"])
cek("Positif (jumlah · %)", kelas["Positif"], T["positif"], toleransi=0.005)
cek("Non-Positif (jumlah · %)", kelas["Non-Positif"], T["non_positif"], toleransi=0.005)
cek("gambar tersimpan", "ya" if gambar.exists() else "tidak", "ya")
selesai()

## Tahap 4 — *Pre-processing*
**Dasar:** Bab III tahap 4; Bab II 2.2 (Tabel 2.1–2.7); `REVISI_BRODY.md` R13–R15.

- **Sel 1/2** membentuk kamus pemanjangan kata Brody dari seluruh ulasan (hanya frekuensi kata; label tidak dipakai), lalu membekukannya ke `hasil/tahap04_kamus_brody.json`.
- **Sel 2/2** menjalankan enam langkah: NFKC, hapus emoji/simbol, huruf kecil, normalisasi pemanjangan kata, tanda baca berulang, dan rapikan spasi. Setelah itu teks kosong dibuang, dan hasilnya disimpan ke `hasil/tahap04_bersih.parquet` (kolom baru `text_clean`).

Tanpa *stemming*, tanpa *stopword removal*, dan tanpa normalisasi kamus slang.

In [ ]:
# ══ Tahap 4 — Pre-processing (sel 1/2: kamus pemanjangan kata Brody) ══
# Package: importlib, collections (bawaan Python); pandas; lib.wondr_teks (sebelum_kamus, kamus_brody, ringkas, KATA)
import importlib
from collections import Counter, defaultdict
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T, ya = TARGET["4"], (lambda x: "ya" if x else "tidak")
M = lambda w: bool(wt.HURUF.search(w))                      # kata memanjang: ada huruf berulang >= 3

# 1) Kamus dari teks langkah 1-3 SELURUH ulasan (R14): hanya frekuensi kata; label dan rating tidak dibaca
masuk = PATH["hasil"] / "tahap03_label.parquet"
pra = pd.read_parquet(masuk, columns=["ulasan"])["ulasan"].astype(str).map(wt.sebelum_kamus)
KAMUS, info = wt.kamus_brody(pra, CFG["brody_ambang"])
K2, i2 = wt.kamus_brody(pra, CFG["brody_ambang"])          # dibentuk ulang: harus identik

# 2) Rincian per kelompok (untuk tabel dan cek): frekuensi semua bentuk, kata memanjang di teks dan jalurnya
frek = Counter(w for t in pra for w in wt.KATA.findall(t))
grup = defaultdict(dict)
for w, n in frek.items():
    grup[wt.ringkas(w)][w] = n
mem = [w for t in pra for w in wt.KATA.findall(t) if M(w)]
lk, cad = [w for w in mem if wt.ringkas(w) in KAMUS], [w for w in mem if wt.ringkas(w) not in KAMUS]
nb = [len(set(x)) for x in (mem, lk, cad)]                  # bentuk unik: total, lewat kamus, lewat cadangan
def baris(k, v):                                            # satu baris tabel kamus
    pm = sorted(((w, n) for w, n in grup[k].items() if M(w)), key=lambda x: (-x[1], x[0]))
    return {"kanonik": v, "frekuensi_kanonik": frek[v], "n_bentuk_memanjang": len(pm),
            "kemunculan_memanjang": sum(n for _, n in pm), "contoh": ", ".join(w for w, _ in pm[:3])}
tab = (pd.DataFrame([baris(k, v) for k, v in KAMUS.items()])
         .sort_values(["kemunculan_memanjang", "kanonik"], ascending=[False, True], ignore_index=True))
seri = sum(sorted(grup[k].values(), reverse=True)[:2].count(frek[v]) > 1 for k, v in KAMUS.items())

# 3) Simpan kamus beku (dipakai sel 2/2 dan ulasan baru) dan tabelnya
simpan_json({"ambang": CFG["brody_ambang"], "info": info, "kamus": KAMUS}, PATH["hasil"] / "tahap04_kamus_brody.json")
keluar = simpan_tabel(tab, "tahap04_kamus_brody")

# 4) Ringkasan + cek
judul("4", "Pre-processing (sel 1/2: kamus Brody)", masukan=f"{rel(masuk)} ({fmt_id(len(pra))} ulasan; langkah 1–3)",
      keluaran=[f"hasil/tahap04_kamus_brody.json ({fmt_id(len(KAMUS))} entri)", rel(keluar)])
print(f"Kamus (ambang frekuensi {CFG['brody_ambang']}): kelompok memanjang {info['kelompok_memanjang']} · "
      f"dipakai {len(KAMUS)} · dibuang: frekuensi < {CFG['brody_ambang']} {info['buang_frekuensi']}, "
      f"kanonik 1 huruf {info['buang_1_huruf']}, kanonik memanjang {info['buang_kanonik_memanjang']}")
print(f"Kata memanjang: {fmt_id(len(mem))} kemunculan ({nb[0]} bentuk) = lewat kamus {len(lk)} ({nb[1]}) "
      f"+ cadangan >= 3 -> 2 {len(cad)} ({nb[2]})")
print(f"\n{'kanonik':<12}{'frekuensi':>10}{'bentuk':>8}{'kemunculan':>12}  contoh bentuk memanjang")
for r in tab.head(12).itertuples():
    print(f"{r.kanonik:<12}{fmt_id(r.frekuensi_kanonik):>10}{r.n_bentuk_memanjang:>8}{r.kemunculan_memanjang:>12}  {r.contoh}")
cek("isi kamus (info · dipakai)", {**info, "dipakai": len(KAMUS)}, T["kamus"])
cek("kata memanjang: total · kamus · cadangan · bentuk", [len(mem), len(lk), len(cad), tuple(nb)],
    [T["memanjang"][k] for k in ("total", "kamus", "cadangan", "bentuk")])
cek("lewat kamus + cadangan = total (kemunculan · bentuk)", [ya(len(lk) + len(cad) == len(mem)), ya(nb[1] + nb[2] == nb[0])],
    ["ya", "ya"])
V = list(KAMUS.values())
cek("kanonik: > 1 huruf · tanpa huruf berulang >= 3 · frekuensi >= ambang · seri", [ya(all(len(v) > 1 for v in V)),
    ya(not any(map(M, V))), ya(all(frek[v] >= CFG["brody_ambang"] for v in V)), seri], ["ya", "ya", "ya", 0])
cek("kamus dibentuk ulang identik (deterministik) · JSON = kamus", [ya(K2 == KAMUS and i2 == info),
    ya(baca_json(PATH["hasil"] / "tahap04_kamus_brody.json")["kamus"] == KAMUS)], ["ya", "ya"])
selesai()

In [ ]:
# ══ Tahap 4 — Pre-processing (sel 2/2: enam langkah dengan kamus beku) ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (langkah, bersihkan, ringkas; memakai re, unicodedata)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["4"]

# 1) Kamus beku dari sel 1/2; uji bersihkan() dengan contoh Bab II (Tabel 2.1-2.7), masukan ditulis escape Unicode
KB = baca_json(PATH["hasil"] / "tahap04_kamus_brody.json")
KAMUS = KB["kamus"]
CONTOH = ["\U0001D401\U0001D400\U0001D406\U0001D414\U0001D412\U0001D412\U0001D412 bgt!!! \u2605\u2605 tp "
          "kadang\u00b2 error, nunggu lamaaa", "\uff4d\uff41\uff4e\uff54\uff41\uff50"]
uji = [wt.bersihkan(x, KAMUS) for x in CONTOH]

# 2) Baca masukan, jalankan 6 langkah berurutan sambil menghitung ulasan yang berubah
masuk = PATH["hasil"] / "tahap03_label.parquet"
df = pd.read_parquet(masuk)
s, berubah = df["ulasan"].astype(str), []
for i, (_, f) in enumerate(wt.langkah(KAMUS)):
    t = s.map(f)
    berubah.append(int((t != s).sum()))
    if i == 3:                                              # langkah 4: simpan contoh sebelum -> sesudah
        contoh4 = pd.DataFrame({"id": df["review_id"], "a": s, "b": t})[t != s].head(3)
    s = t
df["text_clean"] = s

# 3) Buang teks kosong (jumlah sebelum dan sesudah ditampilkan), lalu simpan
kosong, bersih = df[df["text_clean"] == ""], df[df["text_clean"] != ""].reset_index(drop=True)
keluar = PATH["hasil"] / "tahap04_bersih.parquet"
bersih.to_parquet(keluar, index=False)

# 4) Ringkasan + cek
judul("4", "Pre-processing (sel 2/2: enam langkah)", masukan=[f"{rel(masuk)} ({fmt_id(len(df))} baris)",
      f"hasil/tahap04_kamus_brody.json ({fmt_id(len(KAMUS))} entri)"],
      keluaran=f"{rel(keluar)} ({fmt_id(len(bersih))} baris, kolom baru: text_clean)")
print(f"{'Langkah':<40}{'Ulasan berubah':>15}")
for (nama, _), n in zip(wt.langkah(KAMUS), berubah):
    print(f"{nama:<40}{fmt_id(n):>15}")
r_kosong = {int(k): int(v) for k, v in kosong["rating"].value_counts().items()}
print(f"\nTeks kosong dibuang: {fmt_id(len(kosong))} ({', '.join(f'rating {k}: {v}' for k, v in r_kosong.items())})"
      f"  →  {fmt_id(len(df))} menjadi {fmt_id(len(bersih))} ulasan")
print("Contoh langkah 4 saja (kamus Brody, sisanya >= 3 -> 2); tanda baca berulang baru dipadatkan di langkah 5:")
lewat = lambda x: ("kamus" if any(wt.ringkas(w) in KAMUS for w in wt.KATA.findall(x) if wt.HURUF.search(w))
                   else "cadangan")                         # jalur kata memanjang di langkah 4
for i, a, b in contoh4.itertuples(index=False):                # teks utuh, tanpa pemotongan
    kata = "; ".join(f'"{x}" → "{y}" ({lewat(x)})' for x, y in zip(a.split(), b.split()) if x != y)
    print(f"  [{i}] kata yang berubah : {kata}")
    print(f"       hasil 6 langkah  : {bersih.loc[bersih['review_id'] == i, 'text_clean'].iloc[0]}")
cek("kamus beku: ambang · entri", [KB["ambang"], len(KAMUS)], [CFG["brody_ambang"], T["kamus"]["dipakai"]])
cek("uji contoh Bab II (Tabel 2.1-2.7)", uji, T["contoh_bab2"])
cek("ulasan berubah per langkah", berubah, T["berubah"])
cek("teks kosong dibuang", len(kosong), T["kosong"])
cek("rating teks kosong", r_kosong, T["kosong_rating"])
cek("jumlah setelah pre-processing", len(bersih), T["hasil"])
tc = bersih["text_clean"]                                      # invarian: aturan berlaku di seluruh teks
cek("sisa huruf berulang · tanda baca berulang · emoji", [int(tc.map(lambda x: bool(p.search(x))).sum())
    for p in (wt.HURUF, wt.BACA, wt.EMO)], [0, 0, 0])
sama = (bersih["ulasan"].map(lambda x: wt.bersihkan(x, KAMUS)) == tc).all()
cek("text_clean = bersihkan(ulasan, KAMUS) di semua baris", "ya" if sama else "tidak", "ya")
selesai()

## Tahap 5 — Deduplikasi
**Dasar:** Bab III tahap 5; Bab II 2.4.1, Tabel 2.11; `REVISI_30OKT.md` R9.

- **Deduplikasi:** ulasan dengan kunci deduplikasi yang sama digabung menjadi satu dokumen berlabel mayoritas, dan kunci seri dikeluarkan. Keluaran: `hasil/tahap05_dedup.parquet` dan `bab4/tabel/tahap05_alur.csv`.
- **Cek kunci gabungan** *(program saja)* menampilkan kunci yang menggabungkan bentuk teks berbeda. Keluaran: `bab4/tabel/tahap05_kunci_gabungan.csv`.

In [ ]:
# ══ Tahap 5 — Deduplikasi ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (kunci_dedup, deduplikasi)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["5"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap04_bersih.parquet"
d = pd.read_parquet(masuk)

# 2) Deduplikasi (kode acuan): label mayoritas, kunci seri dikeluarkan, wakil = teks tersering lalu id terkecil
hasil, suara, seri = wt.deduplikasi(d)
keluar = PATH["hasil"] / "tahap05_dedup.parquet"
hasil.to_parquet(keluar, index=False)

# 3) Rincian alur, dihitung dari tabel suara per kunci; simpan CSV (angka mentah)
kunci, sah = d["text_clean"].map(wt.kunci_dedup), suara[~seri]
ALUR = {"ulasan masuk": len(d), "kunci kosong (dibuang)": int((kunci == "").sum()),
        "kunci deduplikasi": len(suara), "kunci seri": int(seri.sum()),
        "dibuang: salinan berlabel sama": int((sah[["n_np", "n_p"]].max(axis=1) - 1).sum()),
        "dibuang: anggota kalah suara": int(sah[["n_np", "n_p"]].min(axis=1).sum()),
        "dibuang: anggota kunci seri": int(suara.loc[seri, ["n_np", "n_p"]].to_numpy().sum()),
        "dokumen hasil": len(hasil)}
alur = simpan_tabel(pd.DataFrame({"langkah": list(ALUR), "jumlah": list(ALUR.values())}), "tahap05_alur")

# 4) Ringkasan + cek
dibuang = [len(d) - ALUR["kunci kosong (dibuang)"] - len(hasil)] + [v for k, v in ALUR.items() if k.startswith("dibuang")]
pos = (d["label"].mean() * 100, hasil["label"].mean() * 100)
judul("5", "Deduplikasi", masukan=f"{rel(masuk)} ({fmt_id(len(d))} baris)",
      keluaran=[f"{rel(keluar)} ({fmt_id(len(hasil))} baris, kolom baru: dedup_key)", rel(alur)])
for k, v in ALUR.items():
    print(f"{k:<34}{fmt_id(v):>7}")
print(f"Proporsi Positif                  {fmt_id(pos[0], 1)}% → {fmt_id(pos[1], 1)}%")
print("\nKunci seri yang dikeluarkan (program saja; rating anggotanya saling bertentangan):")
anggota = d.assign(k=kunci)[kunci.isin(suara[seri].index)].sort_values("review_id")
for k, g in anggota.groupby("k"):
    print(f"  {k:<20}" + " · ".join(f"[{i}] rating {r}" for i, r in zip(g["review_id"], g["rating"])))
cek("kunci kosong dibuang", ALUR["kunci kosong (dibuang)"], 0)
cek("kunci · kunci seri · ulasan seri", (len(suara), int(seri.sum()), dibuang[3]),
    (T["kunci"], T["kunci_seri"], T["ulasan_seri"]))
cek("jumlah dokumen", len(hasil), T["dokumen"])
cek("dibuang (total · salinan · kalah suara · seri)", dibuang, T["dibuang"])
cek("total dibuang = jumlah ketiga rincian", "ya" if dibuang[0] == sum(dibuang[1:]) else "tidak", "ya")
cek("Positif (%) sebelum · sesudah", pos, T["positif_pct"], toleransi=0.05)
cek("satu dokumen per kunci", "ya" if hasil["dedup_key"].is_unique else "tidak", "ya")
selesai()

In [ ]:
# ══ Tahap 5 — Cek kunci yang menggabungkan bentuk berbeda (tambahan; program saja, data tidak diubah) ══
# Package: importlib, re (bawaan Python); pandas; lib.wondr_teks (kunci_dedup)
import importlib, re
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T, ya = TARGET["5"], (lambda x: "ya" if x else "tidak")

# 1) Dua kunci per ulasan: k_asli = a-z, 0-9, spasi TANPA peringkasan karakter; k_dedup = kunci_dedup() (dengan peringkasan)
masuk = PATH["hasil"] / "tahap04_bersih.parquet"
d = pd.read_parquet(masuk, columns=["review_id", "text_clean"])
d["k_asli"] = d["text_clean"].map(lambda s: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip())
d["k_dedup"] = d["text_clean"].map(wt.kunci_dedup)
d = d[d["k_dedup"] != ""]

# 2) Kunci terdampak = k_dedup dengan > 1 k_asli berbeda; bentuk diurutkan menurut jumlah ulasan (menurun), lalu abjad
bentuk = d.groupby(["k_dedup", "k_asli"]).size().rename("n").reset_index()
n_bentuk = bentuk.groupby("k_dedup").size()
gab = (bentuk[bentuk["k_dedup"].isin(n_bentuk[n_bentuk > 1].index)]
       .sort_values(["k_dedup", "n", "k_asli"], ascending=[True, False, True]))
gab["teks"] = gab["k_asli"] + " (" + gab["n"].astype(str) + ")"
tab = (gab.groupby("k_dedup", sort=True).agg(n_bentuk=("k_asli", "size"), n_ulasan=("n", "sum"), bentuk=("teks", " | ".join))
          .reset_index().rename(columns={"k_dedup": "kunci"}))
tab["memuat_angka"] = tab["kunci"].str.contains(r"\d")
keluar = simpan_tabel(tab, "tahap05_kunci_gabungan")

# 3) Ringkasan + cek (keputusan penggabungan tetap di tangan peneliti; tidak ada yang diubah otomatis)
judul("5", "Cek kunci yang menggabungkan bentuk berbeda", masukan=f"{rel(masuk)} ({fmt_id(len(d))} ulasan berkunci)",
      keluaran=f"{rel(keluar)} ({len(tab)} kunci)")
print(f"Kunci terdampak: {len(tab)} dari {fmt_id(d['k_dedup'].nunique())} kunci · "
      f"memuat angka: {int(tab['memuat_angka'].sum())} · ulasan di kunci terdampak: {fmt_id(int(tab['n_ulasan'].sum()))}")
print(f"\n{'kunci':<26}bentuk (jumlah ulasan)")
for r in tab.itertuples():
    print(f"{r.kunci:<26}{r.bentuk}")
cek("kunci terdampak · memuat angka", [len(tab), int(tab["memuat_angka"].sum())], list(T["gabungan"]))
cek("jumlah kunci = Tahap 5 · kunci_dedup(k_asli) = k_dedup", [d["k_dedup"].nunique(),
    ya((d["k_asli"].map(wt.kunci_dedup) == d["k_dedup"]).all())], [T["kunci"], "ya"])
selesai()

## Tahap 6.1 — Pembagian data
**Dasar:** Bab III tahap 6; pers. (2.6); Tabel 2.12–2.13.

Pembagian berbasis grup (teks berkunci sama selalu masuk bagian yang sama), dengan stratifikasi label grup × kelompok panjang dan *seed* 42. Proporsinya latih 70% · validasi 15% · uji 15%. Keluaran: `hasil/tahap06_{latih,validasi,uji}.parquet`.

In [ ]:
# ══ Tahap 6.1 — Pembagian data ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (bagi_data, kunci_grup, kelompok_panjang)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["6.1"]
KOLOM = ["review_id", "ulasan", "text_clean", "rating", "label", "group_key", "dedup_key", "kelompok_panjang"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap05_dedup.parquet"
dd = pd.read_parquet(masuk)

# 2) Bagi berbasis grup pers. (2.6), stratifikasi label grup x kelompok panjang (kode acuan; seed dari CFG)
bagi, grup = wt.bagi_data(dd, seed=CFG["seed_bagi"])

# 3) Simpan tiga bagian
keluar = {b: PATH["hasil"] / f"tahap06_{b}.parquet" for b in CFG["proporsi"]}
for b, p in keluar.items():
    bagi.loc[bagi["split"] == b, KOLOM].reset_index(drop=True).to_parquet(p, index=False)

# 4) Ringkasan + cek
hit = {b: (int((bagi.split == b).sum()), int(((bagi.split == b) & (bagi.label == 0)).sum()),
           int(((bagi.split == b) & (bagi.label == 1)).sum())) for b in CFG["proporsi"]}
judul("6.1", "Pembagian data", masukan=f"{rel(masuk)} ({fmt_id(len(dd))} dokumen)",
      keluaran=[f"{rel(p)} ({fmt_id(hit[b][0])} dokumen)" for b, p in keluar.items()])
print(f"Grup: {fmt_id(len(grup))} dari {fmt_id(len(dd))} dokumen "
      f"({fmt_id(int((grup.n > 1).sum()))} grup berisi lebih dari satu dokumen)\n")
print(f"{'Bagian':<10}{'Dokumen':>8}{'Porsi':>8}{'Target':>8}{'Non-Positif':>13}{'Positif':>9}")
for b, p in CFG["proporsi"].items():
    n, n0, n1 = hit[b]
    print(f"{b:<10}{fmt_id(n):>8}{fmt_id(n / len(bagi) * 100, 1) + '%':>8}{fmt_id(p * 100, 0) + '%':>8}"
          f"{fmt_id(n0):>13}{fmt_id(n1):>9}")
cek("PROPORSI di lib/wondr_teks.py = CFG", wt.PROPORSI, CFG["proporsi"])
cek("jumlah grup", len(grup), T["grup"])
for b in CFG["proporsi"]:
    cek(f"{b} (dokumen · Non-Positif · Positif)", hit[b], T[b])
cek("semua dokumen mendapat satu bagian", "ya" if bagi["split"].notna().all() and len(bagi) == len(dd) else "tidak", "ya")
selesai()

## Tahap 6.2 — Cek kebocoran, keseimbangan, dan dampak kamus
**Dasar:** Bab III tahap 6; Bab II 2.4.2–2.4.4; `REVISI_BRODY.md` R16.

- **Sel 1/2:** tidak ada `group_key`, `dedup_key`, atau `text_clean` yang muncul di lebih dari satu bagian.
- **Sel 2/2:** proporsi label dan kelompok panjang per bagian, beserta uji khi-kuadrat. Keluaran: `bab4/tabel/tahap06_strata.csv`.
- **Cek dampak kamus Brody** *(program saja)*: membandingkan teks hasil kamus seluruh data dengan kamus yang dibentuk dari data latih saja. Keluaran: `bab4/tabel/tahap06_dampak_kamus.csv`.

**Mulai di sini data uji dikunci.**

In [ ]:
# ══ Tahap 6.2 — Cek kebocoran dan keseimbangan (sel 1/2: kebocoran dan kelengkapan) ══
# Package: pandas (olah tabel, baca .parquet)
import pandas as pd
mulai()
T, BAGIAN = TARGET["6.2"], list(CFG["proporsi"])

# 1) Baca ketiga bagian dari berkas Tahap 6.1 (pembagian tidak diubah) dan hasil deduplikasi
masuk = {b: PATH["hasil"] / f"tahap06_{b}.parquet" for b in BAGIAN}
df = pd.concat([pd.read_parquet(p).assign(bagian=b) for b, p in masuk.items()], ignore_index=True)
dedup = pd.read_parquet(PATH["hasil"] / "tahap05_dedup.parquet")

# 2) Kebocoran: kunci yang muncul di lebih dari satu bagian
KUNCI = ("group_key", "dedup_key", "text_clean")
lintas = {k: df.groupby(k)["bagian"].nunique() for k in KUNCI}
bocor = {k: s[s > 1] for k, s in lintas.items()}

# 3) Ringkasan + cek
judul("6.2", "Cek kebocoran (sel 1/2)", masukan=[f"{rel(p)} ({fmt_id(int((df.bagian == b).sum()))} dokumen)"
                                                for b, p in masuk.items()])
print(f"{'Kunci':<12}{'Nilai unik':>11}{'Di > 1 bagian':>15}")
for k in KUNCI:
    print(f"{k:<12}{fmt_id(len(lintas[k])):>11}{fmt_id(len(bocor[k])):>15}")
for k in KUNCI:                                             # contoh hanya dicetak bila ada kebocoran
    for kunci in bocor[k].index[:5]:
        print(f"  BOCOR {k} {kunci!r}: {sorted(df.loc[df[k] == kunci, 'bagian'].unique())}")
grup_per_bagian = int(df.groupby("bagian")["group_key"].nunique().sum())
for k in KUNCI:
    cek(f"{k} di lebih dari satu bagian", len(bocor[k]), T["bocor"])
cek("jumlah grup per bagian dijumlahkan", grup_per_bagian, TARGET["6.1"]["grup"])
cek("total dokumen · review_id unik", (len(df), int(df["review_id"].nunique())),
    (TARGET["5"]["dokumen"], TARGET["5"]["dokumen"]))
sama = set(df["review_id"]) == set(dedup["review_id"])
cek("review_id = review_id hasil deduplikasi", "ya" if sama else "tidak", "ya")
selesai()

In [ ]:
# ══ Tahap 6.2 — Cek kebocoran dan keseimbangan (sel 2/2: keseimbangan dan strata) ══
# Package: pandas (olah tabel); scipy.stats.chi2_contingency (uji khi-kuadrat)
import pandas as pd
from scipy.stats import chi2_contingency
mulai()
T, BAGIAN, PANJANG = TARGET["6.2"], list(CFG["proporsi"]), list(CFG["kelompok_panjang"])

# 1) Baca ketiga bagian (pembagian tidak diubah)
df = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet").assign(bagian=b) for b in BAGIAN],
               ignore_index=True)

# 2) Tabel silang bagian x label dan bagian x kelompok panjang, lalu uji khi-kuadrat
x_label = pd.crosstab(df["bagian"], df["label"]).loc[BAGIAN]
x_panjang = pd.crosstab(df["bagian"], df["kelompok_panjang"]).loc[BAGIAN, PANJANG]
p_label, p_panjang = chi2_contingency(x_label)[1], chi2_contingency(x_panjang)[1]
pct_np = x_label[0] / x_label.sum(axis=1) * 100
pct_pj = x_panjang.div(x_panjang.sum(axis=1), axis=0) * 100

# 3) Simpan tabel strata (label x kelompok panjang) per bagian, angka mentah
strata = pd.crosstab([df["label"].map(CFG["kelas"]), df["kelompok_panjang"]], df["bagian"])[BAGIAN]
keluar = simpan_tabel(strata.reset_index().rename(columns={"label": "kelas"}), "tahap06_strata")

# 4) Ringkasan + cek
judul("6.2", "Cek keseimbangan (sel 2/2)", masukan="hasil/tahap06_{latih,validasi,uji}.parquet", keluaran=rel(keluar))
print(f"{'Bagian':<10}{'Non-Positif':>12}" + "".join(f"{'kata ' + k:>12}" for k in PANJANG))
for b in BAGIAN:
    sel = [fmt_id(pct_np[b], 1)] + [fmt_id(pct_pj.loc[b, k], 1) for k in PANJANG]
    print(f"{b:<10}" + "".join(f"{x + '%':>12}" for x in sel))
print(f"\nKhi-kuadrat: bagian x label p = {fmt_id(p_label)} · bagian x kelompok panjang p = {fmt_id(p_panjang)}")
print(f"\nDokumen per strata\n{strata.to_string()}")
cek("Non-Positif (%) latih · validasi · uji", [pct_np[b] for b in BAGIAN], T["non_positif_pct"], toleransi=0.05)
for b in BAGIAN:
    cek(f"kelompok panjang (%) {b}", list(pct_pj.loc[b]), T["panjang_pct"][b], toleransi=0.05)
cek("p khi-kuadrat label · panjang", (p_label, p_panjang), (T["p_label"], T["p_panjang"]), toleransi=5e-5)
print("\nMulai di sini data uji dikunci: label dan metriknya tidak dipakai sampai Tahap 14 "
      "(kecuali teksnya diekspor ke lembar anotasi di Tahap 9.1).")
selesai()

In [ ]:
# ══ Tahap 6.2 — Cek dampak kamus Brody (tambahan; program saja, data tidak diubah) ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (kunci_dedup, sebelum_kamus, kamus_brody, bersihkan,
#          pemanjangan)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T, BAGIAN, ya = TARGET["6.2"]["dampak"], list(CFG["proporsi"]), (lambda x: "ya" if x else "tidak")

# 1) Kamus seluruh data (Tahap 4, beku), ulasan Tahap 4 (sebelum deduplikasi), dan dokumen ketiga bagian (Tahap 6.1)
KAMUS = baca_json(PATH["hasil"] / "tahap04_kamus_brody.json")["kamus"]
bersih = pd.read_parquet(PATH["hasil"] / "tahap04_bersih.parquet", columns=["review_id", "ulasan", "text_clean"])
df = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet", columns=["review_id", "ulasan", "text_clean",
                "dedup_key"]).assign(bagian=b) for b in BAGIAN], ignore_index=True)

# 2) Kamus latih: ulasan yang kunci deduplikasinya ada di data latih (salinan ikut dihitung); label tidak dipakai
lat = bersih[bersih["text_clean"].map(wt.kunci_dedup).isin(set(df.loc[df["bagian"] == "latih", "dedup_key"]))]
KL, info_L = wt.kamus_brody(lat["ulasan"].astype(str).map(wt.sebelum_kamus), CFG["brody_ambang"])
beda_kanonik = {k: (v, KAMUS.get(k)) for k, v in KL.items() if KAMUS.get(k) != v}

# 3) Teks hasil kedua kamus per dokumen; rincian kata memanjang yang hasilnya berbeda (semua bagian)
df["a"], df["b"] = (df["ulasan"].map(lambda s, K=K: wt.bersihkan(s, K)) for K in (KAMUS, KL))
beda = df[df["a"] != df["b"]]
rinci = pd.DataFrame([{"bagian": r.bagian, "review_id": r.review_id, "kata_asli": w,
                       "hasil_kamus_seluruh": wt.pemanjangan(w, KAMUS), "hasil_kamus_latih": wt.pemanjangan(w, KL)}
                      for r in beda.itertuples() for w in wt.KATA.findall(wt.sebelum_kamus(r.ulasan))
                      if wt.pemanjangan(w, KAMUS) != wt.pemanjangan(w, KL)])
keluar = simpan_tabel(rinci, "tahap06_dampak_kamus")

# 4) Ringkasan + cek
n_beda = {b: int((beda["bagian"] == b).sum()) for b in BAGIAN}
judul("6.2", "Cek dampak kamus Brody (program saja)", keluaran=f"{rel(keluar)} ({len(rinci)} baris)",
      masukan=["hasil/tahap04_kamus_brody.json", "hasil/tahap04_bersih.parquet", "hasil/tahap06_{latih,validasi,uji}.parquet"])
print(f"Kamus seluruh data: {len(KAMUS)} entri · kamus latih: {len(KL)} entri (ulasan latih {fmt_id(len(lat))}; "
      f"kelompok memanjang {info_L['kelompok_memanjang']}, dibuang karena frekuensi {info_L['buang_frekuensi']})")
print("Dokumen yang teksnya berbeda: " + " · ".join(f"{b} {n_beda[b]} dari {fmt_id(int((df['bagian'] == b).sum()))}"
                                                    for b in BAGIAN))
for k, (v, w) in beda_kanonik.items():                      # hanya dicetak bila ada (dilaporkan, bukan error)
    print(f"  kanonik berbeda: kelompok {k!r}: kamus latih {v!r} · kamus seluruh data {w!r}")
print(f"\n{'bagian':<10}{'review_id':>10}  {'kata asli':<18}{'kamus seluruh':<16}kamus latih")
for r in rinci[rinci["bagian"] != "latih"].itertuples():
    print(f"{r.bagian:<10}{r.review_id:>10}  {r.kata_asli:<18}{r.hasil_kamus_seluruh:<16}{r.hasil_kamus_latih}")
print("\nTeks yang dipakai model tetap memakai kamus seluruh data; tabel ini hanya mengukur dampaknya.")
cek("text_clean berkas pembagian = bersihkan(ulasan, KAMUS)", ya((df["a"] == df["text_clean"]).all()), "ya")
cek("entri kamus latih dengan kanonik berbeda / tidak ada di kamus seluruh data", len(beda_kanonik), 0)
cek("ulasan latih · kamus latih", [len(lat), {**info_L, "dipakai": len(KL)}], [T["ulasan_latih"], T["kamus_latih"]])
cek("dokumen berbeda per bagian", n_beda, T["berbeda"])
VU = ("validasi", "uji")
cek("review_id berbeda: validasi · uji", [sorted(beda.loc[beda["bagian"] == b, "review_id"].tolist()) for b in VU],
    [T["berbeda_id"][b] for b in VU])
selesai()

## Tahap 7 — Tokenisasi
**Dasar:** Bab III tahap 7; pers. (2.7)–(2.9); Tabel 2.14.

- **Sel 1/2:** menghitung token subkata IndoBERT, *fertility*, r_UNK, dan L_max = 32·⌈maks latih / 32⌉ = **160**, semuanya dari data latih saja. Keluaran: `hasil/tahap07_tokenisasi.json`.
- **Sel 2/2:** histogram panjang token data latih. Keluaran: `bab4/gambar/tahap07_panjang_token.png`.

In [ ]:
# ══ Tahap 7 — Tokenisasi (sel 1/2: token subkata, statistik data latih, L_max) ══
# Package: math (bawaan Python); numpy; pandas; transformers.AutoTokenizer (tokenizer IndoBERT)
import math
import numpy as np, pandas as pd
from transformers import AutoTokenizer
mulai()
T, BAGIAN = TARGET["7"], list(CFG["proporsi"])

# 1) Baca ketiga bagian dan tokenizer bawaan checkpoint (harus sama dengan tahap prapelatihan)
data = {b: pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet") for b in BAGIAN}
tok = AutoTokenizer.from_pretrained(CFG["checkpoint"])

# 2) Token subkata per dokumen (tanpa [CLS]/[SEP]); panjang dokumen = subkata + 2
sub = {b: [tok.tokenize(t) for t in d["text_clean"]] for b, d in data.items()}
panjang = {b: np.array([len(s) + 2 for s in sub[b]]) for b in BAGIAN}
cocok = all((panjang[b] == [len(x) for x in tok(list(data[b]["text_clean"]))["input_ids"]]).all() for b in BAGIAN)

# 3) Statistik HANYA dari data latih: fertility (2.7), r_UNK (2.8), L_max (2.9); validasi dan uji hanya deskripsi
n_sub, n_kata = sum(map(len, sub["latih"])), int(data["latih"]["text_clean"].str.split().str.len().sum())
n_unk = sum(s.count(tok.unk_token) for s in sub["latih"])
F, r_unk, P_q = n_sub / n_kata, n_unk / n_sub, float(np.percentile(panjang["latih"], CFG["q"]))
L_max = CFG["b"] * math.ceil(P_q / CFG["b"])
potong = {b: data[b].loc[panjang[b] > L_max, "review_id"].tolist() for b in BAGIAN}
stat = {b: {"n": len(p), "median": float(np.median(p)), "p95": float(np.percentile(p, 95)), "maks": int(p.max()),
            "terpotong": len(potong[b])} for b, p in panjang.items()}
keluar = simpan_json({"checkpoint": CFG["checkpoint"], "kosakata": tok.vocab_size, "n_subkata": n_sub,
                      "n_kata": n_kata, "n_unk": n_unk, "F": F, "r_UNK": r_unk, "P_q": P_q, "L_max": L_max,
                      "statistik": stat, "terpotong_id": potong, "panjang_latih": panjang["latih"].tolist()},
                     PATH["hasil"] / "tahap07_tokenisasi.json")

# 4) Ringkasan + cek
judul("7", "Tokenisasi (sel 1/2)", masukan=["hasil/tahap06_{latih,validasi,uji}.parquet", f"tokenizer {CFG['checkpoint']}"],
      keluaran=rel(keluar))
print(f"{'Bagian':<10}{'n':>7}{'Median':>8}{'P95':>7}{'Maks':>6}{'Terpotong':>11}")
for b, s in stat.items():
    print(f"{b:<10}{fmt_id(s['n']):>7}{fmt_id(s['median'], 0):>8}{fmt_id(s['p95'], 0):>7}{s['maks']:>6}{s['terpotong']:>11}")
print(f"\nData latih: F = {fmt_id(F)} subkata/kata · r_UNK = {fmt_id(r_unk, 6)} ({fmt_id(n_unk)} dari {fmt_id(n_sub)} token)")
print(f"L_max = {CFG['b']}·⌈P{CFG['q']} / {CFG['b']}⌉ = {CFG['b']}·⌈{fmt_id(P_q, 0)} / {CFG['b']}⌉ = {L_max}")
cek("panjang = subkata + 2 (sesuai tokenizer)", "ya" if cocok else "tidak", "ya")
cek("L_max (pers. 2.9)", L_max, T["L_max"])
cek("dokumen terpotong", {b: len(potong[b]) for b in BAGIAN}, T["terpotong"])
cek("review_id yang terpotong", potong["latih"] + potong["validasi"] + potong["uji"], T["terpotong_id"])
selesai()

In [ ]:
# ══ Tahap 7 — Tokenisasi (sel 2/2: histogram panjang token data latih) ══
# Package: numpy; matplotlib (grafik); lib.bantu (baca_json, simpan_gambar)
import numpy as np
import matplotlib.pyplot as plt
mulai()

# 1) Baca hasil sel 1/2
masuk = PATH["hasil"] / "tahap07_tokenisasi.json"
tk = baca_json(masuk)
p, L_max = np.array(tk["panjang_latih"]), tk["L_max"]

# 2) Histogram (satu seri: tanpa legenda) dengan garis L_max
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(p, bins=range(0, L_max + 9, 4), color="#2a78d6", rwidth=0.85, zorder=2)
ax.axvline(L_max, color="#52514e", linewidth=1.2, zorder=3)
ax.text(L_max - 2, ax.get_ylim()[1] * 0.92, f"L_max = {L_max}", ha="right", fontsize=9, color="#52514e")
ax.set(xlabel="Panjang token per dokumen latih (termasuk [CLS] dan [SEP])", ylabel="Jumlah dokumen")
ax.yaxis.set_major_formatter(lambda y, _: fmt_id(int(y)))
ax.grid(axis="y", color="#e4e3df", linewidth=0.8, zorder=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(colors="#52514e", length=0)
gambar = simpan_gambar(fig, "tahap07_panjang_token")

# 3) Ringkasan + cek
judul("7", "Tokenisasi (sel 2/2)", masukan=rel(masuk), keluaran=rel(gambar))
cek("dokumen latih pada histogram", len(p), TARGET["6.1"]["latih"][0])
cek("semua dokumen latih <= L_max", int((p > L_max).sum()), TARGET["7"]["terpotong"]["latih"])
cek("gambar tersimpan", "ya" if gambar.exists() else "tidak", "ya")
selesai()

# C. Label (Tahap 8–9)
**Sudah final; jangan dijalankan ulang.** Kandidat label salah dideteksi dengan *confident learning*, lalu dianotasi manual oleh dua anotator.

## Tahap 8.1 — Kode pelatihan dan uji cepat
**Dasar:** Bab II 2.6.2 dan 2.8; pers. (2.14), (2.15), (2.18)–(2.22); Tabel 3.1.

- **Sel 1/2** menulis `lib/wondr_latih.py`, yang berisi dua fungsi:
  - `latih()`: IndoBERT, AdamW, jadwal *learning rate* linear dengan *warmup*, *clipping* norma gradien, dan *early stopping* berdasarkan *Macro F1* validasi;
  - `fitness()`: rata-rata *Macro F1* validasi terbaik *seed* 42 dan 43.
- **Sel 2/2** *(GPU)* menjalankan uji cepat: T = 996 dan T_w = 99, lalu dua pelatihan kecil dengan *seed* sama harus identik. Hasil uji tidak disimpan.

> Pada Langkah 3, `lib/wondr_latih.py` ditambah AMP (R18) lalu uji cepat dijalankan ulang.

In [ ]:
%%writefile lib/wondr_latih.py
# lib/wondr_latih.py -- fine-tuning IndoBERT (Tahap 8.1; KT-5, KT-6, KT-7 disetujui 21-10-2026).
# Isinya fungsi saja. Konfigurasi diambil dari CFG lewat pakai_konfig(CFG) di setiap sel pelatihan.
import gc, json, math, time
from pathlib import Path
import numpy as np, pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from lib.bantu import atur_seed

_AKAR = Path(__file__).resolve().parents[1]          # folder kerja .../skripsi_wondr_v3
_K = {}                                              # konfigurasi aktif (salinan CFG + L_max)
_CACHE = {}                                          # tokenizer dan input_ids per review_id, per sesi
TANPA_WD = ("bias", "LayerNorm.weight")              # KT-5: tanpa weight decay, seperti BERT asli


def pakai_konfig(cfg):
    """Salin CFG dan ambil L_max dari hasil Tahap 7; cache tokenisasi dikosongkan bila L_max/checkpoint berubah."""
    L = json.loads((_AKAR / "hasil" / "tahap07_tokenisasi.json").read_text(encoding="utf-8"))["L_max"]
    if (_K.get("checkpoint"), _K.get("L_max")) != (cfg["checkpoint"], L):
        _CACHE.clear()
    _K.clear(); _K.update(cfg); _K["L_max"] = L
    return L


def jadwal(n_latih, wr):
    """Pers. (2.21)-(2.22): T = ceil(n_latih / B) * E dan T_w = floor(wr * T)."""
    T = math.ceil(n_latih / _K["B"]) * _K["E"]
    return T, math.floor(wr * T)


def _tok():
    if "tok" not in _CACHE:
        _CACHE["tok"] = AutoTokenizer.from_pretrained(_K["checkpoint"])
    return _CACHE["tok"]


def _kode(df):
    """input_ids per dokumen dengan truncation=True, max_length=L_max; dihitung sekali per review_id."""
    ids = _CACHE.setdefault("ids", {})
    baru = [(r, t) for r, t in zip(df["review_id"], df["text_clean"]) if r not in ids]
    if baru:
        hasil = _tok()([t for _, t in baru], truncation=True, max_length=_K["L_max"])["input_ids"]
        ids.update({r: x for (r, _), x in zip(baru, hasil)})
    return [ids[r] for r in df["review_id"]]


def _muat(ids, y, acak, seed=None):
    """DataLoader dengan padding dinamis per batch (kanan, token [PAD]); batch = B; num_workers = 0."""
    pad = _tok().pad_token_id

    def kumpul(batch):
        n = max(len(a) for a, _ in batch)
        x = torch.full((len(batch), n), pad, dtype=torch.long)
        m = torch.zeros((len(batch), n), dtype=torch.long)
        for i, (a, _) in enumerate(batch):
            x[i, :len(a)] = torch.tensor(a)
            m[i, :len(a)] = 1
        return x, m, torch.tensor([b for _, b in batch], dtype=torch.long)

    g = torch.Generator().manual_seed(seed) if acak else None
    return DataLoader(list(zip(ids, y)), batch_size=_K["B"], shuffle=acak, generator=g, collate_fn=kumpul, num_workers=0)


@torch.no_grad()
def _peluang(model, ids, dev):
    """Peluang kelas Positif (softmax) per dokumen, urutan tetap."""
    model.eval()
    p = [torch.softmax(model(input_ids=x.to(dev), attention_mask=m.to(dev)).logits, dim=-1)[:, 1].cpu()
         for x, m, _ in _muat(ids, [0] * len(ids), acak=False)]
    return torch.cat(p).double().numpy()


def tebak(p1):
    """argmax dua kelas dari peluang Positif; seri (0,5) -> kelas 0."""
    return (np.asarray(p1) > 0.5).astype(int)


def parameter_baru():
    """Parameter yang tidak ada di checkpoint saat model klasifikasi dibuat (KT-6: harus hanya lapisan klasifikasi)."""
    _, info = AutoModelForSequenceClassification.from_pretrained(_K["checkpoint"], num_labels=2, output_loading_info=True)
    return sorted(info["missing_keys"]), sorted(info["unexpected_keys"])


def latih(df_latih, lr, wr, wd, seed, df_val=None, early_stopping=True, prediksi=None, maks_epoch=None):
    """Satu kali fine-tuning (Bab II 2.8). maks_epoch hanya untuk uji cepat; bawaan = E."""
    if not _K:
        raise RuntimeError("Panggil wl.pakai_konfig(CFG) lebih dulu.")
    if early_stopping and df_val is None:
        raise ValueError("early_stopping=True memerlukan df_val.")
    t0, dev = time.perf_counter(), torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ids_latih, prediksi = _kode(df_latih), prediksi or {}
    ids_val = _kode(df_val) if df_val is not None else None
    ids_pred = {k: _kode(d) for k, d in prediksi.items()}

    atur_seed(seed)                                  # sebelum model dibuat: bobot awal W_c, b_c bergantung seed
    model = AutoModelForSequenceClassification.from_pretrained(_K["checkpoint"], num_labels=2).to(dev)
    grup = [{"params": [p for n, p in model.named_parameters() if not n.endswith(TANPA_WD)], "weight_decay": wd},
            {"params": [p for n, p in model.named_parameters() if n.endswith(TANPA_WD)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(grup, lr=lr, betas=tuple(_K["adam_betas"]), eps=_K["adam_eps"])     # pers. (2.18)
    T, T_w = jadwal(len(df_latih), wr)
    sch = get_linear_schedule_with_warmup(opt, T_w, T)                  # pers. (2.20); T tetap walau berhenti dini
    muat = _muat(ids_latih, df_latih["label"].astype(int).tolist(), acak=True, seed=seed)

    f1_epoch, terbaik, ep_terbaik, tanpa_naik, prob = [], -1.0, None, 0, {}
    for ep in range(1, (maks_epoch or _K["E"]) + 1):
        model.train()
        for x, m, y in muat:
            loss = model(input_ids=x.to(dev), attention_mask=m.to(dev), labels=y.to(dev)).loss   # (2.15), tanpa bobot kelas
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), _K["v"])                         # pers. (2.19)
            opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        if early_stopping:
            f1 = float(f1_score(df_val["label"].astype(int), tebak(_peluang(model, ids_val, dev)),
                                average="macro", zero_division=0))
            f1_epoch.append(f1)
            if f1 > terbaik:                         # diperbarui hanya bila lebih besar
                terbaik, ep_terbaik, tanpa_naik = f1, ep, 0
                prob = {k: _peluang(model, i, dev) for k, i in ids_pred.items()}   # peluang di epoch terbaik
            else:
                tanpa_naik += 1
                if tanpa_naik >= _K["p"]:
                    break
    if not early_stopping:                           # CL: peluang dari bobot epoch terakhir
        prob = {k: _peluang(model, i, dev) for k, i in ids_pred.items()}

    del model, opt, sch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"f1_terbaik": terbaik if early_stopping else None, "epoch_terbaik": ep_terbaik, "f1_per_epoch": f1_epoch,
            "detik": time.perf_counter() - t0, "prob": prob, "T": T, "T_w": T_w, "perangkat": dev.type}


def fitness(lr, wr, wd):
    """Pers. (2.25): rata-rata f1_terbaik seed 42 dan 43; data Tahap 9.3, label = label_akhir (KT-7)."""
    baca = lambda b: pd.read_parquet(_AKAR / "hasil" / f"tahap09_{b}.parquet").assign(label=lambda d: d["label_akhir"])
    df_latih, df_val = baca("latih"), baca("validasi")
    hasil = {s: latih(df_latih, lr, wr, wd, s, df_val=df_val) for s in _K["seed_fitness"]}
    skor = [hasil[s]["f1_terbaik"] for s in _K["seed_fitness"]]
    return {"fitness": float(np.mean(skor)), "skor": skor, "hasil": hasil}

In [ ]:
# ══ Tahap 8.1 — Uji cepat lib/wondr_latih.py (sel 2/2; hasil tidak disimpan) ══
# Package: importlib (bawaan Python); numpy; pandas; torch; lib.wondr_latih
import importlib
import numpy as np, pandas as pd, torch
import lib.wondr_latih as wl
importlib.reload(wl)                                        # pakai isi lib/wondr_latih.py yang terbaru
mulai()
T = TARGET["8.1"]
L_max = wl.pakai_konfig(CFG)

# 1) Jadwal untuk n dokumen latih (Tahap 6.1) dan skema default (pers. 2.21-2.22)
T_, Tw_ = wl.jadwal(TARGET["6.1"]["latih"][0], CFG["default"]["wr"])
# 2) Parameter baru saat model klasifikasi dibuat (KT-6: hanya lapisan klasifikasi W_c, b_c)
baru, sisa = wl.parameter_baru()
# 3) Pelatihan kecil: 64 dokumen latih + 64 validasi, 1 epoch, skema default, dua kali dengan seed yang sama
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet").head(64)
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet").head(64)
d, s = CFG["default"], CFG["seed_fitness"][0]
MB = lambda: torch.cuda.memory_allocated() / 2**20 if torch.cuda.is_available() else 0.0
mem, r = [MB()], []
for _ in range(2):                                          # memori dicatat setelah setiap pelatihan
    r.append(wl.latih(lat, d["lr"], d["wr"], d["wd"], s, df_val=val, prediksi={"val": val}, maks_epoch=1))
    mem.append(MB())
if torch.cuda.is_available():                              # lepas ruang kerja cuBLAS (2 x 32 MiB + 1 MiB cuBLASLt)
    torch._C._cuda_clearCublasWorkspaces()
mem.append(MB())

judul("8.1", "Uji cepat lib/wondr_latih.py", keluaran="(tidak ada; hasil uji tidak disimpan)",
      masukan=["lib/wondr_latih.py", "64 dokumen tahap06_latih + 64 tahap06_validasi"])
print(f"L_max = {L_max} · jadwal n = {fmt_id(TARGET['6.1']['latih'][0])}: T = {T_} · T_w = {Tw_}")
print(f"Parameter baru  : {', '.join(baru)} ({len(baru)} tensor)")
print(f"Tidak terpakai  : {len(sisa)} tensor checkpoint (kepala prapelatihan)")
print(f"Memori GPU (MB) : awal {fmt_id(mem[0], 1)} · setelah pelatihan 1 {fmt_id(mem[1], 1)} · setelah pelatihan 2 "
      f"{fmt_id(mem[2], 1)} · setelah ruang kerja cuBLAS dilepas {fmt_id(mem[3], 1)}")
for i, x in enumerate(r, 1):
    print(f"Pelatihan {i}: lr {d['lr']} · wr {fmt_id(d['wr'], 2)} · wd {fmt_id(d['wd'], 2)} · seed {s} · epoch terbaik "
          f"{x['epoch_terbaik']} · Macro F1 val {fmt_id(x['f1_terbaik'])} · {fmt_id(x['detik'], 1)} detik · {x['perangkat']}")
cek(f"T · T_w (n = {fmt_id(TARGET['6.1']['latih'][0])}, rho = {fmt_id(CFG['default']['wr'], 1)})", (T_, Tw_),
    (T["T"], T["T_w"]))
cek("parameter baru hanya lapisan klasifikasi", baru, ["classifier.bias", "classifier.weight"])
ya, p = (lambda x: "ya" if x else "tidak"), r[0]["prob"]["val"]
cek("dua pelatihan identik (F1 · peluang)", [ya(r[0]["f1_terbaik"] == r[1]["f1_terbaik"]),
    ya(np.array_equal(p, r[1]["prob"]["val"]))], ["ya", "ya"])
cek("peluang di [0, 1] untuk 64 dokumen", ya(len(p) == 64 and ((p >= 0) & (p <= 1)).all()), "ya")
cek("pelatihan di GPU", r[0]["perangkat"], "cuda")
cek("memori GPU tidak bertambah antar-pelatihan (MB)", mem[2] - mem[1], 1, cara="<=")
cek("sisa memori setelah ruang kerja cuBLAS dilepas (MB)", mem[3] - mem[0], 1, cara="<=")
selesai()

## Tahap 8.2 — *Confident learning*: pelatihan 5 lipatan *(GPU)*
**Dasar:** Bab III tahap 8; Bab II 2.3.4; Northcutt dkk. (2021).

- **Sel 1/2:** data latih dibagi menjadi 5 lipatan berbasis grup. Untuk setiap lipatan, model skema *default* (*seed* 42, 4 *epoch*) dilatih pada 4 lipatan lain, lalu memprediksi lipatan yang ditahan dan data validasi. Keluaran: `log/cl_lipatan_k.parquet`. Lipatan yang sudah tersimpan dilewati.
- **Sel 2/2:** memeriksa kelengkapan peluang tanpa GPU.

Data uji tidak dipakai.

In [ ]:
# ══ Tahap 8.2 — Confident learning: pelatihan 5 lipatan (sel 1/2) ══
# Package: os, importlib (bawaan Python); numpy; pandas; scikit-learn (StratifiedGroupKFold, f1_score); lib.wondr_latih
import os, importlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
import lib.wondr_latih as wl
importlib.reload(wl)                                        # pakai isi lib/wondr_latih.py yang terbaru
mulai()
wl.pakai_konfig(CFG)
d, K, s = CFG["default"], CFG["cl_lipatan"], CFG["cl_seed_latih"]

# 1) Baca data latih dan validasi (label awal); data uji tidak dibaca
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet")
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet")

# 2) Lipatan berbasis grup (pers. 2.6), terstratifikasi label awal
lipatan = np.empty(len(lat), dtype=int)
sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=CFG["cl_seed_lipatan"])
for k, (_, idx) in enumerate(sgkf.split(lat, lat["label"], groups=lat["group_key"])):
    lipatan[idx] = k

# 3) Setiap lipatan k: latih pada 4 lipatan lain (skema default, E epoch, tanpa early stopping), lalu prediksi
#    lipatan k dan seluruh data validasi. Satu berkas per lipatan; lipatan yang sudah tersimpan dilewati.
for k in range(K):
    berkas = PATH["log"] / f"cl_lipatan_{k}.parquet"
    tr, te = lat[lipatan != k], lat[lipatan == k]
    if berkas.exists():
        print(f"Lipatan {k}: sudah tersimpan di {rel(berkas)}, dilewati")
        continue
    r = wl.latih(tr, d["lr"], d["wr"], d["wd"], s, early_stopping=False, prediksi={"latih": te, "validasi": val})
    out = pd.concat([pd.DataFrame({"review_id": x["review_id"].to_numpy(), "bagian": b, "p1": r["prob"][b]})
                     for b, x in (("latih", te), ("validasi", val))], ignore_index=True)
    out.assign(lipatan=k, n_latih=len(tr), detik=r["detik"]).to_parquet(berkas.with_suffix(".tmp"), index=False)
    os.replace(berkas.with_suffix(".tmp"), berkas)            # tulis atomik: aman bila Colab terputus
    f1 = f1_score(te["label"], wl.tebak(r["prob"]["latih"]), average="macro", zero_division=0)
    print(f"Lipatan {k}: lr {d['lr']} · wr {fmt_id(d['wr'], 2)} · wd {fmt_id(d['wd'], 2)} · seed {s} · {CFG['E']} epoch · "
          f"latih {fmt_id(len(tr))} · prediksi {fmt_id(len(te))} + {fmt_id(len(val))} · Macro F1 lipatan {fmt_id(f1)} · "
          f"{fmt_id(r['detik'] / 60, 1)} menit")
selesai()

In [ ]:
# ══ Tahap 8.2 — Confident learning: ringkasan dan cek 5 lipatan (sel 2/2) ══
# Package: numpy; pandas; scikit-learn (StratifiedGroupKFold, f1_score); lib.wondr_latih (tebak)
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
import lib.wondr_latih as wl
mulai()
K = CFG["cl_lipatan"]

# 1) Baca data latih/validasi dan kelima berkas lipatan; lipatan disusun ulang untuk mencocokkan berkas
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet")
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet")
cl = pd.concat([pd.read_parquet(PATH["log"] / f"cl_lipatan_{k}.parquet") for k in range(K)], ignore_index=True)
lipatan = np.empty(len(lat), dtype=int)
sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=CFG["cl_seed_lipatan"])
for k, (_, idx) in enumerate(sgkf.split(lat, lat["label"], groups=lat["group_key"])):
    lipatan[idx] = k
oof = cl[cl.bagian == "latih"].merge(lat[["review_id", "label", "group_key"]], on="review_id", how="left")
f1 = lambda g: f1_score(g["label"], wl.tebak(g["p1"]), average="macro", zero_division=0)

# 2) Ringkasan per lipatan (Macro F1 terhadap label awal, deskriptif) + cek
judul("8.2", "Confident learning: 5 lipatan", masukan="hasil/tahap06_latih.parquet, hasil/tahap06_validasi.parquet",
      keluaran=f"log/cl_lipatan_0..{K - 1}.parquet ({fmt_id(len(cl))} baris)")
print(f"{'k':<3}{'n latih':>9}{'n lipatan':>11}{'n validasi':>12}{'Non-Positif':>13}{'Macro F1':>10}{'Durasi':>11}")
for k, g in oof.groupby("lipatan"):
    m = cl[cl.lipatan == k]
    print(f"{k:<3}{fmt_id(int(m.n_latih.iloc[0])):>9}{fmt_id(len(g)):>11}{fmt_id(int((m.bagian == 'validasi').sum())):>12}"
          f"{fmt_id((g.label == 0).mean() * 100, 1) + '%':>13}{fmt_id(f1(g)):>10}{fmt_id(m.detik.iloc[0] / 60, 1) + ' mnt':>11}")
garis()
print(f"Macro F1 di luar sampel, seluruh data latih (label awal, deskriptif): {fmt_id(f1(oof))} · "
      f"total durasi {fmt_id(cl.groupby('lipatan').detik.first().sum() / 60, 1)} menit")
ya = lambda x: "ya" if x else "tidak"
cek("dokumen latih dengan tepat 1 peluang di luar sampel", [int(oof.review_id.nunique()), ya(oof.review_id.is_unique)],
    [len(lat), "ya"])
cek("lipatan di berkas = lipatan yang disusun ulang", ya(oof[["review_id", "lipatan"]].merge(pd.DataFrame(
    {"review_id": lat.review_id, "k": lipatan}), on="review_id", how="outer").eval("lipatan == k").all()), "ya")
cek("group_key terbelah antarlipatan", int((oof.groupby("group_key").lipatan.nunique() > 1).sum()), 0)
cek("dokumen validasi dengan tepat 5 peluang", int((cl[cl.bagian == "validasi"].groupby("review_id").lipatan.nunique()
    == K).sum()), len(val))
cek("peluang di [0, 1] · tanpa NaN", [ya(cl.p1.between(0, 1).all()), ya(cl.p1.notna().all())], ["ya", "ya"])
cek("review_id di luar latih/validasi (data uji tidak disentuh)",
    len(set(cl.review_id) - set(lat.review_id) - set(val.review_id)), 0)
selesai()

## Tahap 8.3 — Ambang, *confident joint*, dan kandidat
**Dasar:** Bab III tahap 8; Bab II 2.3.4, pers. (2.3)–(2.4), Tabel 2.9.

- **Sel 1/2** menulis `lib/wondr_label.py` (rumus ditulis sendiri, tanpa `cleanlab`).
- **Sel 2/2** menguji fungsi dengan contoh Tabel 2.9, lalu menerapkannya terpisah pada data latih dan validasi. Keluaran: `hasil/tahap08_kandidat.parquet` dan `bab4/tabel/tahap08_cj.csv`.

In [ ]:
%%writefile lib/wondr_label.py
# lib/wondr_label.py -- confident learning dua kelas (Tahap 8.3). Rumus ditulis sendiri sesuai pers. (2.3)-(2.4),
# Bab II subbab 2.3.4; pustaka cleanlab tidak dipakai. Isinya fungsi saja.
import math
import numpy as np, pandas as pd


def confident_learning(p1, y):
    """Ambang kelas, kelas taksiran, confident joint, dan kandidat label salah; p0 = 1 - p1.

    t_j      = rata-rata p_j pada dokumen berlabel j                                   (pers. 2.3)
    taksiran = argmax p_l di antara kelas l dengan p_l >= t_l; -1 bila tidak ada yang memenuhi ambang
               (dokumen itu tidak dihitung); bila p0 = p1 dan keduanya lolos, dipilih kelas 0
    C[i, j]  = banyak dokumen berlabel i dengan kelas taksiran j; baris = label, kolom = taksiran (pers. 2.4)
    kandidat = dokumen yang dihitung dengan kelas taksiran != label (unsur di luar diagonal C)
    """
    p1, y = np.asarray(p1, dtype=float), np.asarray(y)
    if not (set(np.unique(y)) == {0, 1} and np.all((p1 >= 0) & (p1 <= 1))):
        raise ValueError("y harus memuat kelas 0 dan 1 saja, dan p1 harus di [0, 1] tanpa NaN.")
    y = y.astype(int)
    P = np.column_stack([1.0 - p1, p1])                       # kolom j = peluang kelas j
    t = np.array([math.fsum(P[y == j, j]) / (y == j).sum() for j in (0, 1)])   # pers. (2.3); jumlah dibulatkan tepat
    lolos = P >= t
    taksiran = np.where(lolos.any(axis=1), np.where(lolos, P, -np.inf).argmax(axis=1), -1)
    dihitung = taksiran >= 0
    C = np.zeros((2, 2), dtype=int)
    np.add.at(C, (y[dihitung], taksiran[dihitung]), 1)        # pers. (2.4)
    return {"t": t, "taksiran": taksiran, "dihitung": dihitung, "C": C, "kandidat": dihitung & (taksiran != y)}


def per_bagian(d):
    """confident_learning() terpisah untuk setiap bagian (Bab III tahap 8); d berkolom review_id, bagian, label, p1.

    Mengembalikan (tabel per dokumen: review_id, bagian, label, p1, kelas_taksiran [<NA> = tidak dihitung], kandidat;
    tabel per bagian: n, t0, t1, tidak_dihitung, C00, C01, C10, C11, kandidat, kandidat_pct)."""
    dok, cj = [], []
    for b, x in d.groupby("bagian", sort=False):
        r = confident_learning(x["p1"], x["label"])
        taksiran = pd.array(np.where(r["dihitung"], r["taksiran"], pd.NA), dtype="Int64")
        dok.append(x[["review_id", "bagian", "label", "p1"]].assign(kelas_taksiran=taksiran, kandidat=r["kandidat"]))
        cj.append({"bagian": b, "n": len(x), "t0": r["t"][0], "t1": r["t"][1], "tidak_dihitung": int((~r["dihitung"]).sum()),
                   **{f"C{i}{j}": int(r["C"][i, j]) for i in (0, 1) for j in (0, 1)}, "kandidat": int(r["kandidat"].sum())})
    cj = pd.DataFrame(cj)
    cj["kandidat_pct"] = 100 * cj["kandidat"] / cj["n"]
    return pd.concat(dok, ignore_index=True), cj

In [ ]:
# ══ Tahap 8.3 — Ambang, confident joint, dan kandidat (sel 2/2) ══
# Package: importlib (bawaan Python); numpy; pandas; lib.wondr_label (confident_learning, per_bagian)
import importlib
import numpy as np, pandas as pd
import lib.wondr_label as wlab
importlib.reload(wlab)                                      # pakai isi lib/wondr_label.py yang terbaru
mulai()
T, K, n_uji, ya = TARGET["8.3"], CFG["cl_lipatan"], TARGET["6.1"]["uji"][0], (lambda x: "ya" if x else "tidak")

# 1) Peluang Positif: latih = di luar sampel (1 model); validasi = rata-rata 5 model. Data uji tidak dibaca.
cl = pd.concat([pd.read_parquet(PATH["log"] / f"cl_lipatan_{k}.parquet") for k in range(K)])
p = cl.groupby(["bagian", "review_id"], as_index=False).agg(p1=("p1", "mean"), n_model=("p1", "size"))
d = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet", columns=["review_id", "label"]).assign(bagian=b)
               for b in ("latih", "validasi")]).merge(p, on=["bagian", "review_id"], how="left")

# 2) Ambang (2.3), confident joint (2.4), dan kandidat, terpisah untuk latih dan validasi; lalu simpan
dok, cj = wlab.per_bagian(d)
dok.to_parquet(PATH["hasil"] / "tahap08_kandidat.parquet", index=False)
simpan_tabel(cj, "tahap08_cj")

# 3) Ringkasan dan cek; fungsi diuji dengan contoh Tabel 2.9 (d1-d6)
u = wlab.confident_learning([0.95, 0.90, 0.15, 0.10, 0.20, 0.85], [1, 1, 1, 0, 0, 0])
judul("8.3", "Ambang, confident joint, dan kandidat", masukan=f"log/cl_lipatan_0..{K - 1}.parquet, tahap06 latih/validasi",
      keluaran=[f"hasil/tahap08_kandidat.parquet ({fmt_id(len(dok))} baris)", "bab4/tabel/tahap08_cj.csv"])
print(f"{'Bagian':<10}{'n':>6}{'t0':>8}{'t1':>8}{'C00':>7}{'C01 NP→P':>10}{'C10 P→NP':>10}{'C11':>7}{'Kandidat':>15}"
      f"{'Tdk dihitung':>14}")
for c in cj.itertuples():
    print(f"{c.bagian:<10}{fmt_id(c.n):>6}{fmt_id(c.t0):>8}{fmt_id(c.t1):>8}{fmt_id(c.C00):>7}{fmt_id(c.C01):>10}"
          f"{fmt_id(c.C10):>10}{fmt_id(c.C11):>7}{fmt_id(c.kandidat) + ' (' + fmt_id(c.kandidat_pct, 1) + '%)':>15}"
          f"{fmt_id(c.tidak_dihitung):>14}")
print(f"C: baris = label awal, kolom = kelas taksiran · dokumen untuk lembar anotasi (Tahap 9.1): "
      f"{' + '.join(fmt_id(k) for k in cj.kandidat)} + uji {fmt_id(n_uji)} = {fmt_id(int(cj.kandidat.sum()) + n_uji)}")
cek("contoh Tabel 2.9: t0 · t1", u["t"].tolist(), [T["t0"], T["t1"]], toleransi=5e-5)
cek("contoh Tabel 2.9: C · kandidat (d)", [u["C"].tolist(), (np.flatnonzero(u["kandidat"]) + 1).tolist()], [T["C"], T["kandidat"]])
cek("peluang per dokumen: latih 1 model · validasi 5 model", [ya(d.query("bagian == 'latih'").n_model.eq(1).all()),
    ya(d.query("bagian == 'validasi'").n_model.eq(K).all())], ["ya", "ya"])
for c in cj.itertuples():
    cek(f"{c.bagian}: jumlah kandidat", c.kandidat, 0, cara=">")
    cek(f"{c.bagian}: kandidat (%)", c.kandidat_pct, 40, cara="<=", d=1)
    cek(f"{c.bagian}: diagonal C dominan (C00 > C01 · C11 > C10)", [ya(c.C00 > c.C01), ya(c.C11 > c.C10)], ["ya", "ya"])
selesai()

## Tahap 9.1 — Ekspor lembar anotasi
**Dasar:** Bab III tahap 9; Bab II 2.3.5.

- **Sel 1/2** menulis `lib/wondr_anotasi.py`, berisi petunjuk anotasi serta fungsi tulis/baca lembar.
- **Sel 2/2** menyusun lembar dari kandidat latih, kandidat validasi, dan seluruh data uji. Urutannya diacak dengan `default_rng(2026)`. Keluaran: `anotasi/lembar_anotator_{A,B}.xlsx` dan `anotasi/kunci_anotasi.csv`.

Lembar yang sudah diisi tidak akan ditimpa.

In [ ]:
%%writefile lib/wondr_anotasi.py
# lib/wondr_anotasi.py -- lembar anotasi Excel (Tahap 9). Isinya teks petunjuk dan fungsi saja.
import pandas as pd
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.datavalidation import DataValidation

PETUNJUK = [  # Bab III tahap 9; makna kelas Bab II subbab 2.3.2; contoh teks dari Tabel 2.8 (tanpa rating)
    "PETUNJUK ANOTASI SENTIMEN ULASAN APLIKASI WONDR BY BNI",
    "",
    "Tugas: baca setiap teks pada sheet 'Anotasi', lalu isi kolom 'label' dengan 1 atau 0.",
    "  1 = Positif     : ulasan menyatakan kepuasan atau pujian tanpa keluhan.",
    "  0 = Non-Positif : keluhan, kritik, ulasan netral, atau ulasan bernada campuran (memuat pujian sekaligus keluhan).",
    "",
    "Contoh:",
    "  'aplikasi ini sangat membantu, transaksi jadi cepat'  -> 1",
    "  'aplikasi bagus, tetapi notifikasi sering terlambat'  -> 0 (campuran)",
    "  'verifikasi gagal terus padahal data sudah benar'     -> 0 (keluhan)",
    "",
    "Aturan:",
    "  1. Nilai hanya dari isi teks. Teks sudah dibersihkan: huruf kecil, emoji dan simbol dihapus, huruf dan tanda baca",
    "     yang berulang dipendekkan.",
    "  2. Isi sendiri, terpisah dari anotator lain, tanpa bantuan alat AI. Jangan berdiskusi dengan anotator lain",
    "     sampai seluruh lembar selesai.",
    "  3. Isi setiap baris dengan 1 atau 0 saja; jangan ada baris yang kosong.",
    "  4. Jangan mengubah kolom id_anotasi dan teks, urutan baris, maupun nama sheet.",
    "  5. Simpan dengan nama berkas yang sama, lalu serahkan kepada peneliti.",
]


def tulis_lembar(path, data, kolom_isian, petunjuk=None):
    """Tulis xlsx: sheet 'Petunjuk' (opsional) lalu sheet 'Anotasi' = data. Kolom isian dibiarkan seperti di data
    dan diberi validasi Excel 0/1; teks ditulis sebagai string (tidak pernah ditafsirkan sebagai rumus).
    Berkas yang kolom isiannya sudah terisi (hasil kerja anotator) tidak pernah ditimpa."""
    if path.exists() and baca_lembar(path)[kolom_isian].notna().any():
        raise FileExistsError(f"{path.name} sudah berisi isian '{kolom_isian}'; berkas tidak ditimpa.")
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        if petunjuk is not None:
            pd.DataFrame({"p": petunjuk}).to_excel(w, sheet_name="Petunjuk", index=False, header=False)
            w.book["Petunjuk"].column_dimensions["A"].width = 120
        data.to_excel(w, sheet_name="Anotasi", index=False)
        ws, n = w.book["Anotasi"], len(data) + 1
        for j, nama in enumerate(data.columns, start=1):
            huruf = ws.cell(1, j).column_letter
            ws.column_dimensions[huruf].width = 90 if nama == "teks" else 12
            ws.cell(1, j).font = Font(bold=True)
            for i in range(2, n + 1):
                c = ws.cell(i, j)
                if isinstance(c.value, str):
                    c.data_type = "s"                          # cegah teks berawalan '=' menjadi rumus
                if nama == "teks":
                    c.alignment = Alignment(wrap_text=True, vertical="top")
            if nama == kolom_isian:
                dv = DataValidation(type="list", formula1='"0,1"', allow_blank=True, showErrorMessage=True,
                                    errorTitle="Nilai tidak sah", error="Isi dengan 1 (Positif) atau 0 (Non-Positif).")
                dv.add(f"{huruf}2:{huruf}{n}")
                ws.add_data_validation(dv)
        ws.freeze_panes = "A2"


def baca_lembar(path):
    """Baca sheet 'Anotasi' apa adanya: teks tidak diubah menjadi NaN; sel kosong tetap kosong (NaN)."""
    return pd.read_excel(path, sheet_name="Anotasi", dtype={"id_anotasi": str, "teks": str}, keep_default_na=False,
                         na_values=[""])

In [ ]:
# ══ Tahap 9.1 — Ekspor lembar anotasi (sel 2/2) ══
# Package: importlib (bawaan Python); numpy; pandas; openpyxl (lewat lib.wondr_anotasi)
import importlib
import numpy as np, pandas as pd
import lib.wondr_anotasi as wa
importlib.reload(wa)                                        # pakai isi lib/wondr_anotasi.py yang terbaru
mulai()
ya, A = (lambda x: "ya" if x else "tidak"), PATH["anotasi"]

# 1) Kandidat latih dan validasi (Tahap 8.3) + seluruh data uji. Teks = text_clean (KT-10). Dari data uji hanya
#    review_id dan text_clean yang dibaca (tanpa rating dan label).
kd = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet", columns=["review_id", "bagian", "kandidat"])
teks = {b: pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet", columns=["review_id", "text_clean"])
        for b in ("latih", "validasi", "uji")}
sumber = pd.concat([kd[kd.kandidat & (kd.bagian == b)].merge(teks[b], on="review_id").sort_values("review_id")
                    for b in ("latih", "validasi")] + [teks["uji"].assign(bagian="uji", kandidat=False)
                    .sort_values("review_id")], ignore_index=True)        # urutan dasar: bagian, lalu review_id

# 2) Acak dengan default_rng(2026), beri id_anotasi D0001..., simpan kunci (bukan untuk anotator) dan 2 lembar identik
lembar = sumber.iloc[np.random.default_rng(CFG["seed_anotasi"]).permutation(len(sumber))].reset_index(drop=True)
lembar.insert(0, "id_anotasi", [f"D{i:04d}" for i in range(1, len(lembar) + 1)])
lembar[["id_anotasi", "review_id", "bagian", "kandidat"]].to_csv(A / "kunci_anotasi.csv", index=False)
isi = pd.DataFrame({"id_anotasi": lembar.id_anotasi, "teks": lembar.text_clean, "label": None})
for a in "AB":
    wa.tulis_lembar(A / f"lembar_anotator_{a}.xlsx", isi, kolom_isian="label", petunjuk=wa.PETUNJUK)

# 3) Baca ulang kedua lembar dan kunci untuk cek; ringkasan per sumber
L, k = {a: wa.baca_lembar(A / f"lembar_anotator_{a}.xlsx") for a in "AB"}, pd.read_csv(A / "kunci_anotasi.csv")
cj, n = pd.read_csv(PATH["tabel"] / "tahap08_cj.csv").set_index("bagian"), k.bagian.value_counts()[["latih", "validasi", "uji"]]
judul("9.1", "Ekspor lembar anotasi", masukan=["hasil/tahap08_kandidat.parquet", "hasil/tahap06_{latih,validasi,uji}.parquet"],
      keluaran=["anotasi/lembar_anotator_A.xlsx · lembar_anotator_B.xlsx (identik)", "anotasi/kunci_anotasi.csv (tidak untuk anotator)"])
for nama, v in zip(["kandidat latih", "kandidat validasi", "uji (seluruhnya)", "total"], [*n, n.sum()]):
    print(f"{nama:<20}{fmt_id(int(v)):>7}")
cek("jumlah: kandidat latih · kandidat validasi · uji", n.tolist(), [*cj.kandidat[["latih", "validasi"]], TARGET["6.1"]["uji"][0]])
cek("sheet · kolom lembar", [pd.ExcelFile(A / "lembar_anotator_A.xlsx").sheet_names, list(L["A"].columns)],
    [["Petunjuk", "Anotasi"], ["id_anotasi", "teks", "label"]])
cek("lembar A = lembar B · teks = text_clean · label kosong", [ya(L["A"].equals(L["B"])), ya(L["A"].teks.equals(isi.teks)),
    int(L["A"].label.notna().sum())], ["ya", "ya", 0])
cek("id_anotasi = kunci · review_id unik · teks unik", [ya(L["A"].id_anotasi.equals(k.id_anotasi)), int(k.review_id.nunique()),
    int(L["A"].teks.nunique())], ["ya", len(k), len(k)])
selesai()

## Tahap 9.2 — Impor anotasi dan kesepakatan
**Dasar:** Bab III tahap 9; Bab II 2.3.5, pers. (2.5), Tabel 2.10.

- **Sel 1/2** menulis `lib/wondr_kesepakatan.py`.
- **Sel 2/2** memvalidasi kedua lembar dan menghitung κ Cohen sebelum diskusi, secara keseluruhan dan per sumber. Keluaran: `bab4/tabel/tahap09_kappa.csv` dan `anotasi/selisih.xlsx`.

In [ ]:
%%writefile lib/wondr_kesepakatan.py
# lib/wondr_kesepakatan.py -- validasi isian lembar anotasi, selisih, dan kesepakatan dua penilai (Tahap 9.2, 9.3, 14.3).
# Kappa Cohen pers. (2.5); tafsiran Tabel 2.10 (Landis dan Koch, 1977). Isinya teks petunjuk dan fungsi saja.
import numpy as np, pandas as pd
from openpyxl import load_workbook
from sklearn.metrics import cohen_kappa_score
from lib.wondr_anotasi import tulis_lembar

PETUNJUK_SELISIH = [
    "PETUNJUK PENYELESAIAN SELISIH ANOTASI",
    "",
    "Baris pada sheet 'Anotasi' adalah ulasan yang diberi label berbeda oleh anotator A dan B.",
    "Diskusikan setiap baris bersama kedua anotator, lalu isi kolom 'label_final' dengan 1, 0, atau TS:",
    "  1  = Positif     : ulasan menyatakan kepuasan atau pujian tanpa keluhan.",
    "  0  = Non-Positif : keluhan, kritik, ulasan netral, atau ulasan bernada campuran (memuat pujian sekaligus keluhan).",
    "  TS = tetap tidak sepakat setelah diskusi; label akhirnya 0 (Non-Positif) menurut aturan cadangan.",
    "",
    "Jangan mengubah kolom lain, urutan baris, maupun nama sheet; jangan ada label_final yang kosong.",
]
BATAS_KAPPA = [(0.20, "Sedikit (slight)"), (0.40, "Cukup (fair)"), (0.60, "Sedang (moderate)"),
               (0.80, "Substansial (substantial)"), (1.00, "Hampir sempurna (almost perfect)")]


def periksa_isian(x, id_kunci, teks_kunci, kolom):
    """Validasi lembar terisi terhadap kunci: id lengkap, unik, dan sama; teks tidak diubah; isian hanya 0/1 tanpa sel
    kosong. Mengembalikan (ringkasan, isian numerik dalam urutan id_kunci, daftar id bermasalah)."""
    acuan = pd.DataFrame({"id_anotasi": list(id_kunci), "teks_kunci": list(teks_kunci)})
    m = acuan.merge(x, on="id_anotasi", how="left", indicator=True)
    nilai = pd.to_numeric(m[kolom], errors="coerce")
    sah = nilai.isin([0, 1])
    r = {"id_lengkap": bool(x["id_anotasi"].is_unique and len(x) == len(acuan) and (m["_merge"] == "both").all()),
         "teks_sama": bool((m["teks"] == m["teks_kunci"]).all()),
         "kosong": int(m[kolom].isna().sum()),
         "tidak_sah": int((m[kolom].notna() & ~sah).sum())}
    r["valid"] = r["id_lengkap"] and r["teks_sama"] and r["kosong"] == 0 and r["tidak_sah"] == 0
    masalah = m.loc[~sah | (m["teks"] != m["teks_kunci"]) | (m["_merge"] != "both"), "id_anotasi"].tolist()
    return r, nilai.astype("Int64"), masalah


def kappa(a, b):
    """Pers. (2.5): p_o = proporsi label sama; p_e = sum_k p_A,k * p_B,k; kappa = (p_o - p_e) / (1 - p_e)."""
    a, b = np.asarray(a).astype(int), np.asarray(b).astype(int)
    if a.shape != b.shape or a.size == 0 or not np.isin(np.r_[a, b], [0, 1]).all():
        raise ValueError("a dan b harus sepanjang, tidak kosong, dan hanya berisi 0/1.")
    p_o = float(np.mean(a == b))
    p_e = float(sum(np.mean(a == k) * np.mean(b == k) for k in (0, 1)))
    return {"n": int(a.size), "p_o": p_o, "p_e": p_e, "kappa": (p_o - p_e) / (1 - p_e) if p_e < 1 else float("nan")}


def tafsir_kappa(k):
    """Tabel 2.10: kappa < 0 buruk; selang lain dibaca dengan batas atas inklusif (0,20; 0,40; 0,60; 0,80; 1,00)."""
    if np.isnan(k):
        return "tidak terdefinisi"
    if k < 0:
        return "Buruk (poor)"
    return next((nama for batas, nama in BATAS_KAPPA if k <= batas), BATAS_KAPPA[-1][1])


def tabel_kappa(a, b, kelompok):
    """Kappa per kelompok (dict nama -> mask boolean): rumus (2.5) dan sklearn, tafsiran Tabel 2.10, jumlah selisih."""
    a, b, baris = np.asarray(a).astype(int), np.asarray(b).astype(int), []
    for nama, m in kelompok.items():
        m = np.asarray(m, dtype=bool)
        r = kappa(a[m], b[m])
        baris.append({"sumber": nama, **r, "kappa_sklearn": float(cohen_kappa_score(a[m], b[m])),
                      "selisih": int((a[m] != b[m]).sum()), "tafsiran": tafsir_kappa(r["kappa"])})
    return pd.DataFrame(baris)


def ekspor_selisih(path, d):
    """Tulis selisih.xlsx (id_anotasi, teks, label_A, label_B, label_final kosong) untuk baris label_A != label_B, urut
    id_anotasi; validasi Excel label_final = 0, 1, atau TS (REVISI_30OKT 5.2). Berkas yang label_final-nya sudah diisi
    tidak ditimpa (hanya dicatat). Mengembalikan baris selisih."""
    s = d[d["label_A"] != d["label_B"]].sort_values("id_anotasi")
    data = pd.DataFrame({"id_anotasi": s["id_anotasi"], "teks": s["text_clean"], "label_A": s["label_A"],
                         "label_B": s["label_B"], "label_final": None})
    try:
        tulis_lembar(path, data, kolom_isian="label_final", petunjuk=PETUNJUK_SELISIH)
    except FileExistsError as galat:
        print(f"Catatan: {galat}")
        return s
    wb = load_workbook(path)                                  # pilihan isian label_final: 0, 1, atau TS
    for dv in wb["Anotasi"].data_validations.dataValidation:
        dv.formula1, dv.error = '"0,1,TS"', "Isi dengan 1 (Positif), 0 (Non-Positif), atau TS (tetap tidak sepakat)."
    wb.save(path)
    return s

In [ ]:
# ══ Tahap 9.2 — Impor anotasi dan kesepakatan (sel 2/2) ══
# Package: importlib (bawaan Python); numpy; pandas; lib.wondr_anotasi; lib.wondr_kesepakatan (scikit-learn cohen_kappa_score)
import importlib
import numpy as np, pandas as pd
import lib.wondr_anotasi as wa, lib.wondr_kesepakatan as wk
importlib.reload(wk)                                        # pakai isi lib/wondr_kesepakatan.py yang terbaru
mulai()
ya, A, T, e = (lambda x: "ya" if x else "tidak"), PATH["anotasi"], TARGET["9.2"], (lambda v: f"{v:.1e}".replace(".", ","))

# 1) Kunci + text_clean (dari data uji hanya review_id dan text_clean), lalu validasi kedua lembar yang sudah diisi
k = pd.read_csv(A / "kunci_anotasi.csv").merge(pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet",
        columns=["review_id", "text_clean"]) for b in ("latih", "validasi", "uji")]), on="review_id", how="left")
V = {a: wk.periksa_isian(wa.baca_lembar(A / f"lembar_anotator_{a}.xlsx"), k.id_anotasi, k.text_clean, "label") for a in "AB"}
judul("9.2", "Impor anotasi dan kesepakatan", masukan=["anotasi/lembar_anotator_A.xlsx · B.xlsx", "anotasi/kunci_anotasi.csv"],
      keluaran=["bab4/tabel/tahap09_kappa.csv", "anotasi/selisih.xlsx (label_final kosong)"])
cek_lembar = lambda: [cek(f"lembar {a}: id lengkap · teks tidak diubah · sel kosong · isian selain 0/1", [ya(r["id_lengkap"]),
                      ya(r["teks_sama"]), r["kosong"], r["tidak_sah"]], ["ya", "ya", 0, 0]) for a, (r, _, _) in V.items()]
if not all(r["valid"] for r, _, _ in V.values()):
    cek_lembar(); [print(f"  lembar {a}, id bermasalah ({len(m)}): {', '.join(m[:20])}") for a, (_, _, m) in V.items() if m]
    raise ValueError("Lembar anotasi belum valid: perbaiki id di atas pada lembarnya, unggah ulang, lalu jalankan sel ini lagi.")

# 2) Kappa pers. (2.5) dan sklearn per sumber + keseluruhan; jumlah selisih
d = k.assign(label_A=V["A"][1].astype(int).to_numpy(), label_B=V["B"][1].astype(int).to_numpy())
tab = wk.tabel_kappa(d.label_A, d.label_B, {"kandidat latih": d.bagian == "latih", "kandidat validasi": d.bagian == "validasi",
                                            "uji": d.bagian == "uji", "keseluruhan": d.bagian.notna()})
simpan_tabel(tab, "tahap09_kappa")

# 3) Ekspor selisih untuk diskusi (label_final kosong); berkas yang sudah diisi tidak ditimpa
s = wk.ekspor_selisih(A / "selisih.xlsx", d)
sel, u = wa.baca_lembar(A / "selisih.xlsx"), wk.kappa([1] * 45 + [0] * 55, [1] * 40 + [0] * 5 + [1] * 10 + [0] * 45)  # contoh 2.3.5
print(f"{'Sumber':<19}{'n':>6}{'p_o':>8}{'p_e':>8}{'κ':>8}  {'Tafsiran':<33}{'Selisih':>8}")
for t in tab.itertuples():
    print(f"{t.sumber:<19}{fmt_id(t.n):>6}{fmt_id(t.p_o):>8}{fmt_id(t.p_e):>8}{fmt_id(t.kappa):>8}  {t.tafsiran:<33}{fmt_id(t.selisih):>8}")
cek_lembar()
cek("contoh Bab II 2.3.5: p_o · p_e · κ", [u["p_o"], u["p_e"], u["kappa"]], [0.85, 0.50, T["kappa_contoh"]], toleransi=T["toleransi"], d=4)
dk = float((tab.kappa - tab.kappa_sklearn).abs().max())
cek(f"κ rumus (2.5) = sklearn (|Δ| maks {e(dk)} ≤ 1e-12)", ya(dk <= T["toleransi"]), "ya")
cek("selisih.xlsx: baris = selisih keseluruhan · id sama", [len(sel), ya(sel.id_anotasi.tolist() == s.id_anotasi.tolist())],
    [int(tab.selisih.iloc[-1]), "ya"])
selesai()

## Tahap 9.3 — Terapkan label akhir
**Dasar:** Bab III tahap 9; `REVISI_30OKT.md` R8.

- **Sel 1/2** menentukan label manual: label A bila A dan B sepakat, atau `label_final` hasil diskusi bila berbeda (TS → 0, aturan cadangan). Keluaran: `anotasi/label_manual.csv` dan `hasil/tahap09_anotasi.json`.
- **Sel 2/2** membentuk `label_akhir`:
  - latih dan validasi: label manual untuk kandidat, label awal untuk dokumen lain;
  - uji: label manual untuk seluruh dokumen.

  Keluaran: `hasil/tahap09_{latih,validasi,uji}.parquet`.

In [ ]:
# ══ Tahap 9.3 — Terapkan label akhir (sel 1/2: label manual dari dua anotator, diskusi, dan aturan cadangan) ══
# Package: importlib (bawaan Python); pandas; lib.wondr_anotasi; lib.wondr_kesepakatan
import importlib
import pandas as pd
import lib.wondr_anotasi as wa, lib.wondr_kesepakatan as wk
importlib.reload(wk)                                        # pakai isi lib/wondr_kesepakatan.py yang terbaru
mulai()
ya, A = (lambda x: "ya" if x else "tidak"), PATH["anotasi"]

# 1) Kunci + text_clean; validasi ulang lembar A dan B, lalu selisih.xlsx (label_final: 1, 0, atau TS; REVISI 5.2)
k = pd.read_csv(A / "kunci_anotasi.csv").merge(pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet",
        columns=["review_id", "text_clean"]) for b in ("latih", "validasi", "uji")]), on="review_id", how="left")
V = {a: wk.periksa_isian(wa.baca_lembar(A / f"lembar_anotator_{a}.xlsx"), k.id_anotasi, k.text_clean, "label") for a in "AB"}
if not all(r["valid"] for r, _, _ in V.values()):
    raise ValueError("Lembar A/B tidak valid lagi; jalankan sel Tahap 9.2 untuk melihat id yang bermasalah.")
k = k.assign(label_A=V["A"][1].astype(int).to_numpy(), label_B=V["B"][1].astype(int).to_numpy())
s, sel = k[k.label_A != k.label_B], wa.baca_lembar(A / "selisih.xlsx")
asli = sel["label_final"]                                  # isian apa adanya; sel kosong = belum diisi
ts = asli.astype("string").str.strip().str.upper().eq("TS").fillna(False).to_numpy()
n_diskusi, n_cadangan = int(pd.to_numeric(asli, errors="coerce").isin([0, 1]).sum()), int(ts.sum())
sel = sel.assign(label_final=asli.where(~ts, 0))            # aturan cadangan R8: TS (tetap tidak sepakat) -> 0
rs, final, masalah = wk.periksa_isian(sel, s.id_anotasi, s.text_clean, "label_final")
ab = sel.set_index("id_anotasi").reindex(s.id_anotasi)[["label_A", "label_B"]].to_numpy()
ab_tetap = bool((ab == s[["label_A", "label_B"]].to_numpy()).all())
if not (rs["valid"] and ab_tetap):
    print(f"selisih.xlsx bermasalah ({len(masalah)} id): {', '.join(masalah[:20]) or '-'} · label_A/label_B tetap: {ya(ab_tetap)}")
    raise ValueError("Isi label_final 1, 0, atau TS untuk semua baris selisih.xlsx tanpa mengubah kolom lain, "
                     "lalu jalankan lagi.")

# 2) Label manual = label A bila A = B, label_final bila berbeda; simpan bersama kappa (dari Tahap 9.2) ke json
k["label_final"] = pd.Series(final.to_numpy(), index=s.index).reindex(k.index).astype("Int64")
k["label_manual"] = k.label_final.fillna(k.label_A).astype(int)
k["cadangan"] = k.id_anotasi.isin(set(sel.id_anotasi[ts]))  # label_final = 0 karena TS (aturan cadangan)
k[["id_anotasi", "review_id", "bagian", "kandidat", "label_A", "label_B", "label_final", "label_manual",
   "cadangan"]].to_csv(A / "label_manual.csv", index=False)
tab = pd.read_csv(PATH["tabel"] / "tahap09_kappa.csv")
simpan_json({"n_anotasi": len(k), "n_selisih": len(s), "n_diskusi": n_diskusi, "n_cadangan": n_cadangan,
             "kappa": tab.set_index("sumber")[["n", "p_o", "p_e", "kappa", "tafsiran", "selisih"]].to_dict("index")},
            PATH["hasil"] / "tahap09_anotasi.json")
judul("9.3", "Label manual", masukan=["anotasi/lembar_anotator_A.xlsx · B.xlsx", "anotasi/selisih.xlsx"],
      keluaran=["anotasi/label_manual.csv", "hasil/tahap09_anotasi.json"])
print(f"dokumen dianotasi {fmt_id(len(k))} · sepakat {fmt_id(len(k) - len(s))} · selisih {fmt_id(len(s))} (diskusi 0/1: "
      f"{fmt_id(n_diskusi)} · aturan cadangan TS → 0: {fmt_id(n_cadangan)}) · κ keseluruhan {fmt_id(tab.kappa.iloc[-1])} "
      f"({tab.tafsiran.iloc[-1]})")
cek("lembar A · B valid · selisih.xlsx valid · label_A/label_B di selisih tidak berubah",
    [ya(V["A"][0]["valid"]), ya(V["B"][0]["valid"]), ya(rs["valid"]), ya(ab_tetap)], ["ya"] * 4)
cek("n_diskusi + n_cadangan = n_selisih (REVISI 5.2)", n_diskusi + n_cadangan, len(s))
cek("label manual lengkap · hanya 0/1 · selisih di kappa CSV = selisih lembar", [int(k.label_manual.notna().sum()),
    ya(k.label_manual.isin([0, 1]).all()), int(tab.selisih.iloc[-1])], [len(k), "ya", len(s)])
selesai()

In [ ]:
# ══ Tahap 9.3 — Terapkan label akhir (sel 2/2: label_awal, label_akhir, kandidat, berubah) ══
# Package: numpy; pandas
import numpy as np, pandas as pd
mulai()
ya, T6 = (lambda x: "ya" if x else "tidak"), TARGET["6.1"]

# 1) Data Tahap 6.1, kandidat Tahap 8.3, dan label manual (sel 1/2)
m = pd.read_csv(PATH["anotasi"] / "label_manual.csv")
kd = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet", columns=["review_id", "kandidat"])
# 2) latih/validasi: label_akhir = label manual untuk kandidat, label awal untuk lainnya; uji: label manual seluruhnya
D = {}
for b in ("latih", "validasi", "uji"):
    d = pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet").rename(columns={"label": "label_awal"})
    d = d.merge(kd, on="review_id", how="left") if b != "uji" else d.assign(kandidat=False)
    d = d.merge(m.loc[m.bagian == b, ["review_id", "id_anotasi", "label_manual"]], on="review_id", how="left")
    d["label_manual"] = d.label_manual.astype("Int64")
    pakai = d.kandidat if b != "uji" else pd.Series(True, index=d.index)
    d["label_akhir"] = d.label_manual.where(pakai, d.label_awal).astype("int64")   # galat bila ada label manual yang hilang
    d["berubah"] = (d.label_akhir != d.label_awal).astype(bool)
    d.to_parquet(PATH["hasil"] / f"tahap09_{b}.parquet", index=False)
    D[b] = d

judul("9.3", "Terapkan label akhir", masukan=["hasil/tahap06_{latih,validasi,uji}.parquet", "hasil/tahap08_kandidat.parquet",
      "anotasi/label_manual.csv"], keluaran="hasil/tahap09_{latih,validasi,uji}.parquet")
print(f"{'Bagian':<10}{'n':>7}{'kandidat':>10}{'berubah P→NP':>14}{'berubah NP→P':>14}{'Non-Positif sebelum → sesudah':>32}")
for b in ("latih", "validasi"):
    d = D[b]
    print(f"{b:<10}{fmt_id(len(d)):>7}{fmt_id(int(d.kandidat.sum())):>10}{fmt_id(int((d.berubah & (d.label_awal == 1)).sum())):>14}"
          f"{fmt_id(int((d.berubah & (d.label_awal == 0)).sum())):>14}"
          f"{fmt_id((d.label_awal == 0).mean() * 100, 1) + '% → ' + fmt_id((d.label_akhir == 0).mean() * 100, 1) + '%':>32}")
print(f"{'uji':<10}{fmt_id(len(D['uji'])):>7}  dokumen berlabel manual (sebaran label uji belum ditampilkan)")
cek("n latih · validasi · uji", [len(D[b]) for b in D], [T6["latih"][0], T6["validasi"][0], T6["uji"][0]])
cek("label manual tersedia: kandidat latih · kandidat validasi · seluruh uji", [int(D["latih"].label_manual.notna().sum()),
    int(D["validasi"].label_manual.notna().sum()), int(D["uji"].label_manual.notna().sum())],
    [int(D["latih"].kandidat.sum()), int(D["validasi"].kandidat.sum()), T6["uji"][0]])
cek("bukan kandidat tidak berubah (latih · validasi) · label_akhir hanya 0/1 · kolom label → label_awal",
    [int((D["latih"].berubah & ~D["latih"].kandidat).sum()), int((D["validasi"].berubah & ~D["validasi"].kandidat).sum()),
     ya(all(D[b].label_akhir.isin([0, 1]).all() for b in D)), ya(all("label" not in D[b] and "label_awal" in D[b] for b in D))],
    [0, 0, "ya", "ya"])
selesai()

## Tahap 9.4 — Eksplorasi data latih dan validasi *(program saja)*
**Dasar:** Bab III (eksplorasi data). Hanya data latih dan validasi.

- **Sel 1/2:** proporsi label awal vs label akhir, kelompok panjang × label, median jumlah kata, dan kata tersering per kelas. Keluaran: `bab4/tabel/tahap09_eksplorasi_*.csv`.
- **Sel 2/2:** gambar proporsi label serta acuan minimum validasi (`DummyClassifier`). Keluaran: `bab4/gambar/tahap09_proporsi_label.png` dan `hasil/tahap09_acuan.json`.

In [ ]:
# ══ Tahap 9.4 — Eksplorasi data latih dan validasi (sel 1/2: proporsi, panjang, kata tersering; program saja) ══
# Package: re, collections (bawaan Python); pandas; lib.wondr_teks (KATA_FUNGSI, kelompok_panjang)
import re, collections
import pandas as pd
from lib.wondr_teks import KATA_FUNGSI, kelompok_panjang
mulai()
KP, NM, ya = list(CFG["kelompok_panjang"]), {0: "NP", 1: "P"}, (lambda x: "ya" if x else "tidak")
# 1) Data Tahap 9.3, hanya latih dan validasi (data uji tidak dibaca)
D = {b: pd.read_parquet(PATH["hasil"] / f"tahap09_{b}.parquet").assign(n_kata=lambda d: d.text_clean.str.split().str.len())
     for b in ("latih", "validasi")}
# 2) Proporsi label awal vs akhir + jumlah berubah; kelompok panjang x label akhir; 10 kata tersering per kelas (latih)
prop = pd.DataFrame([{"bagian": b, "n": len(d), **{f"{j}_{NM[k]}": int((d[f"label_{j}"] == k).sum()) for j in ("awal", "akhir")
                      for k in (0, 1)}, "berubah_P_ke_NP": int((d.berubah & (d.label_awal == 1)).sum()),
                      "berubah_NP_ke_P": int((d.berubah & (d.label_awal == 0)).sum())} for b, d in D.items()])
pj = pd.concat([pd.crosstab(d.kelompok_panjang, d.label_akhir).reindex(index=KP, columns=[0, 1], fill_value=0)
                .rename(columns=NM).assign(bagian=b, median_kata_NP=d.n_kata[d.label_akhir == 0].median(),
                median_kata_P=d.n_kata[d.label_akhir == 1].median()) for b, d in D.items()]).reset_index()
hit = {k: collections.Counter(w for s in D["latih"].text_clean[D["latih"].label_akhir == k] for w in re.findall(r"[^\W\d_]+", s)
                  if w not in KATA_FUNGSI).most_common(10) for k in (0, 1)}
kata = pd.DataFrame({"peringkat": range(1, 11), **{f"{c}_{NM[k]}": [x[i] for x in hit[k]] for k in (0, 1)
                     for i, c in ((0, "kata"), (1, "frek"))}})
[simpan_tabel(t, f"tahap09_eksplorasi_{nama}") for t, nama in ((prop, "proporsi"), (pj, "panjang"), (kata, "kata"))]

judul("9.4", "Eksplorasi data (sel 1/2)", masukan="hasil/tahap09_{latih,validasi}.parquet",
      keluaran="bab4/tabel/tahap09_eksplorasi_{proporsi,panjang,kata}.csv")
for r in prop.itertuples():
    print(f"{r.bagian:<9} n {fmt_id(r.n)} · Non-Positif {fmt_id(r.awal_NP)} → {fmt_id(r.akhir_NP)} ({fmt_id(100 * r.awal_NP / r.n, 1)}% → "
          f"{fmt_id(100 * r.akhir_NP / r.n, 1)}%) · berubah P→NP {fmt_id(r.berubah_P_ke_NP)}, NP→P {fmt_id(r.berubah_NP_ke_P)}")
garis(); print(f"{'Bagian':<10}{'Kelompok':<10}{'Non-Positif':>12}{'Positif':>9}{'% Non-Positif':>15}")
for r in pj.itertuples():
    print(f"{r.bagian:<10}{r.kelompok_panjang:<10}{fmt_id(r.NP):>12}{fmt_id(r.P):>9}{fmt_id(100 * r.NP / (r.NP + r.P), 1) + '%':>15}")
print("Median kata per kelas:", " · ".join(f"{b} NP {fmt_id(g.median_kata_NP.iloc[0], 1)}, P {fmt_id(g.median_kata_P.iloc[0], 1)}"
                                            for b, g in pj.groupby("bagian", sort=False)))
garis(); print(f"{'#':<4}{'Non-Positif (latih)':<26}{'Positif (latih)':<26}")
for r in kata.itertuples():
    print(f"{r.peringkat:<4}{r.kata_NP + ' (' + fmt_id(r.frek_NP) + ')':<26}{r.kata_P + ' (' + fmt_id(r.frek_P) + ')':<26}")
cek("akhir NP = awal NP + berubah P→NP − berubah NP→P (latih · validasi)", (prop.akhir_NP).tolist(),
    (prop.awal_NP + prop.berubah_P_ke_NP - prop.berubah_NP_ke_P).tolist())
cek("kelompok_panjang Tahap 6 = kelompok dari jumlah kata · total tabel panjang = n", [ya(all((d.n_kata.map(kelompok_panjang)
    == d.kelompok_panjang).all() for d in D.values())), int((pj.NP + pj.P).sum())], ["ya", int(prop.n.sum())])
selesai()

In [ ]:
# ══ Tahap 9.4 — Eksplorasi data (sel 2/2: gambar proporsi label dan acuan minimum validasi; program saja) ══
# Package: numpy; pandas; matplotlib (grafik); scikit-learn (DummyClassifier, accuracy_score, f1_score)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
mulai()
WARNA, ya = {1: "#2a78d6", 0: "#eb6834"}, (lambda x: "ya" if x else "tidak")   # warna kelas sama dengan Tahap 3
# 1) Label awal dan akhir latih + validasi (data uji tidak dibaca)
D = {b: pd.read_parquet(PATH["hasil"] / f"tahap09_{b}.parquet", columns=["label_awal", "label_akhir"]) for b in ("latih", "validasi")}
# 2) Gambar: proporsi kelas sebelum (label awal) dan sesudah anotasi (label akhir), batang bertumpuk 100%
baris = [(f"{b}: label {j}", (D[b][f"label_{j}"] == 0).mean() * 100) for b in D for j in ("awal", "akhir")]
fig, ax = plt.subplots(figsize=(6, 2.6))
for i, (nama, np_) in enumerate(baris):
    ax.barh(i, np_, color=WARNA[0], height=0.6, zorder=2)
    ax.barh(i, 100 - np_, left=np_, color=WARNA[1], height=0.6, zorder=2)
    for x, v in ((np_ / 2, np_), (np_ + (100 - np_) / 2, 100 - np_)):
        ax.text(x, i, f"{fmt_id(v, 1)}%", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
ax.set(yticks=range(len(baris)), yticklabels=[n for n, _ in baris], xlim=(0, 100), xlabel="Proporsi dokumen (%)")
ax.invert_yaxis(); ax.spines[["top", "right", "left"]].set_visible(False); ax.tick_params(colors="#52514e", length=0)
ax.legend(handles=[Patch(color=WARNA[k], label=CFG["kelas"][k]) for k in (0, 1)], frameon=False, ncol=2,
          loc="lower center", bbox_to_anchor=(0.5, 1.0))
gambar = simpan_gambar(fig, "tahap09_proporsi_label")
# 3) Acuan minimum (program saja): DummyClassifier(most_frequent) dilatih pada label_akhir latih, dinilai pada validasi
y_lat, y_val = D["latih"].label_akhir.to_numpy(), D["validasi"].label_akhir.to_numpy()
dm = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_lat), 1)), y_lat)
tebak = dm.predict(np.zeros((len(y_val), 1)))
acu = {"strategi": "most_frequent", "kelas_mayoritas_latih": int(tebak[0]), "akurasi_validasi": accuracy_score(y_val, tebak),
       "macro_f1_validasi": f1_score(y_val, tebak, average="macro", zero_division=0)}
simpan_json(acu, PATH["hasil"] / "tahap09_acuan.json")
q = (y_val == acu["kelas_mayoritas_latih"]).mean()                         # bentuk tertutup: F1 mayoritas = 2q/(1+q)
judul("9.4", "Eksplorasi data (sel 2/2)", masukan="hasil/tahap09_{latih,validasi}.parquet",
      keluaran=["bab4/gambar/tahap09_proporsi_label.png", "hasil/tahap09_acuan.json"])
print(f"Acuan minimum: selalu menebak {CFG['kelas'][acu['kelas_mayoritas_latih']]} (kelas mayoritas latih) · "
      f"akurasi validasi {fmt_id(acu['akurasi_validasi'])} · Macro F1 validasi {fmt_id(acu['macro_f1_validasi'])}")
cek("acuan = bentuk tertutup (akurasi = q · Macro F1 = q/(1+q))", [acu["akurasi_validasi"], acu["macro_f1_validasi"]],
    [q, q / (1 + q)], toleransi=1e-12, d=4)
cek("kelas mayoritas latih · gambar tersimpan", [acu["kelas_mayoritas_latih"], ya(gambar.exists())],
    [int(np.bincount(y_lat).argmax()), "ya"])
selesai()

# D. Pelatihan dan optimasi (Tahap 10–13)

## Tahap 10 — *Fine-tuning* skema *default* *(GPU, ±7,5 menit di T4)*
**Dasar:** Bab III tahap 10; Tabel 3.1; pers. (2.14)–(2.22), (2.25).

Skema *default* (lr 2e-5 · *warmup ratio* 0,1 · *weight decay* 0,01) dilatih dengan *seed* 42 dan 43. *Fitness* = rata-rata *Macro F1* validasi terbaik kedua *seed*. Keluaran: `hasil/tahap10_default.json`. Data uji tidak dibaca.

> **Status:** sudah dijalankan dalam FP32 (*fitness* 0,9276). **Jangan dijalankan ulang sekarang.** Pada Langkah 4, sel ini diganti versi `REVISI_12NOV.md` 6.4 (AMP, peluang validasi, lingkungan). Perkiraan waktunya sudah mengikuti R17 dan anggaran 40 (5 *run*; dua platform vs satu platform; total Tahap 10–13 = 817 pelatihan).

In [ ]:
# ══ Tahap 10 — Fine-tuning skema default ══
# Package: importlib (bawaan Python); numpy; lib.wondr_latih (fitness; torch, transformers, scikit-learn)
import importlib
import numpy as np
import lib.wondr_latih as wl
importlib.reload(wl)                                        # pakai isi lib/wondr_latih.py yang terbaru
mulai()
d, T, S, ya = CFG["default"], TARGET["10"], CFG["seed_fitness"], (lambda x: "ya" if x else "tidak")
wl.pakai_konfig(CFG)
acuan = baca_json(PATH["hasil"] / "tahap09_acuan.json")["macro_f1_validasi"]

# 1) Fitness pers. (2.23), (2.25)-(2.26): rata-rata Macro F1 validasi terbaik seed 42 dan 43; data Tahap 9.3, label_akhir
r = wl.fitness(d["lr"], d["wr"], d["wd"])
H = r["hasil"]

# 2) Simpan skor, kurva per epoch, dan durasi tiap pelatihan (keputusan no. 21)
simpan_json({**d, "T": H[S[0]]["T"], "T_w": H[S[0]]["T_w"], "fitness": r["fitness"], **{f"skor{s}": H[s]["f1_terbaik"] for s in S},
             **{f"detik{s}": H[s]["detik"] for s in S}, "per_seed": {str(s): {k: H[s][k] for k in ("f1_per_epoch", "epoch_terbaik",
             "f1_terbaik", "detik", "perangkat")} for s in S}, "acuan_minimum": acuan}, PATH["hasil"] / "tahap10_default.json")

# 3) Ringkasan: T dan T_w, satu baris per seed, fitness, dan perkiraan waktu Tahap 11 (RS) + 12.1 (PSO) (R17, R20; anggaran 40)
n_run, menit = len(CFG["rs_seeds"]), np.mean([H[s]["detik"] for s in S]) / 60     # run per metode; menit per pelatihan
n_rs, n_pso = n_run * len(S) * CFG["rs_M"], n_run * len(S) * CFG["pso"]["s"] * (1 + CFG["pso"]["G_max"])  # per metode
n_total = len(S) + n_rs + n_pso + 3 * len(CFG["seed_akhir"])                     # Tahap 10-13 (13: 3 skema x 5 seed)
jam = lambda n: fmt_id(n * menit / 60, 1)                                         # n pelatihan -> jam
judul("10", "Fine-tuning skema default", masukan="hasil/tahap09_{latih,validasi}.parquet (label_akhir)",
      keluaran="hasil/tahap10_default.json")
print(f"lr {d['lr']} · wr {fmt_id(d['wr'], 2)} · wd {fmt_id(d['wd'], 2)} · T = {H[S[0]]['T']} · T_w = {H[S[0]]['T_w']}")
for s in S:
    h = H[s]
    print(f"seed {s}: Macro F1 per epoch [{' · '.join(fmt_id(f) for f in h['f1_per_epoch'])}] · epoch terbaik {h['epoch_terbaik']} · "
          f"{fmt_id(h['f1_terbaik'])} · {fmt_id(h['detik'] / 60, 1)} menit · {h['perangkat']}")
print(f"fitness = ({' + '.join(fmt_id(H[s]['f1_terbaik']) for s in S)}) / 2 = {fmt_id(r['fitness'])} · acuan minimum validasi "
      f"{fmt_id(acuan)} · selisih {fmt_id(r['fitness'] - acuan)}")
print(f"Perkiraan Tahap 11 + 12.1: {n_run} run per metode × {len(S)} seed × {CFG['rs_M']} evaluasi · "
      f"{fmt_id(menit, 2)} menit per pelatihan")
print(f"  dua platform : RS di Kaggle {fmt_id(n_rs)} + PSO di Colab {fmt_id(n_pso)} pelatihan "
      f"≈ {jam(max(n_rs, n_pso))} jam (bersamaan)")
print(f"  satu platform: {fmt_id(n_rs + n_pso)} pelatihan ≈ {jam(n_rs + n_pso)} jam · "
      f"total Tahap 10–13: {fmt_id(n_total)} pelatihan")
cek(f"T · T_w (n latih {fmt_id(TARGET['6.1']['latih'][0])}, rho {fmt_id(d['wr'], 1)})", (H[S[0]]["T"], H[S[0]]["T_w"]),
    (T["T"], T["T_w"]))
cek("fitness > acuan minimum validasi", r["fitness"], acuan, cara=">")
cek("epoch terbaik = epoch pertama dengan F1 tertinggi · berhenti sesuai patience · perangkat (seed 42 · 43)",
    [[ya(h["epoch_terbaik"] == 1 + int(np.argmax(h["f1_per_epoch"]))), ya(len(h["f1_per_epoch"]) in (min(CFG["E"],
      h["epoch_terbaik"] + CFG["p"]), CFG["E"])), h["perangkat"]] for h in (H[s] for s in S)], [["ya", "ya", "cuda"]] * len(S))
selesai()

# E. Tahap berikutnya (menyusul)
Sel-sel berikut ditambahkan ke notebook ini satu per satu, sesuai urutan kerja `REVISI_12NOV.md` Bagian 7, setelah output langkah sebelumnya diperiksa.

| Langkah | Sel | Platform |
|---|---|---|
| 2 | Tahap 0.1 dan 0.2: `VERSI`, revisi *checkpoint*, `amp` | Colab |
| 3 | `lib/wondr_latih.py` (AMP), uji cepat 8.1, **Tahap 10.0 Uji AMP** → keputusan AMP | Colab, GPU |
| 4 | Tahap 10 versi baru (diulang bila AMP dipakai) | Colab, GPU |
| 5 | `lib/wondr_optim.py`, `lib/wondr_pso.py`, uji fungsi PSO, Tahap 11 Persiapan | Colab, CPU |
| 6 | **Tahap 12.1 PSO** (5 *run* × 40 evaluasi) | Colab, GPU |
| 7–8 | **Tahap 10.1 Paket Kaggle**, `KAGGLE_PENCARIAN.ipynb`, uji kesetaraan (R20) | Colab + Kaggle |
| 9–10 | Impor *state* Kaggle, cek silang, **Tahap 11 ringkasan RS** | Colab |
| 11 | **Tahap 12.2** ringkasan metode dan Mann–Whitney · **12.3** ragam antar-*seed* | Colab, CPU |
| 12 | **Tahap 13** cek determinisme dan model akhir (15 pelatihan) | Colab, GPU |
| 13 | **Tahap 14.1–14.3** evaluasi · **Tahap 15** uji signifikansi | Colab, CPU |

Sel *random search* lama (10 *run*, 50 kandidat) **tidak dimuat** karena bertentangan dengan R17, R23, dan anggaran 40. Penggantinya, `wo.jalankan_rs`, dibuat pada Langkah 5.

**Pengaman T4** (keputusan pengguna 13-11-2026): sel RS dan PSO yang baru berhenti dengan pesan galat bila GPU sesi bukan T4.